# ODS build v2 (full)

All 53 ods tables in one notebook, one SQL statement per cell. Generated by `ods_build/v2/build_notebook.py` from `entities/*.yaml`: the same SQL as the split `ODS_v2_*` notebooks.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ods_build/v2/entities/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# ODS v2 settings (a pipeline can override these).
target_schema = "ods_v2"
stg_schema    = "stg"
source_set    = "frozen"          # frozen | prod
cutoff_date   = "2021-04-01"   # event / transaction tables start here; policy tables keep all history
batch_id      = ""                # blank = generated from the start time
# The ods hash salt: set it here (read from Key Vault at a later stage). It must never change once ods is in use:
# every hashed key (email, phone, licence, registration, address) depends on it.
hash_salt     = "slkjdghwe234"


In [ ]:
%%pyspark
import datetime, json
SOURCE_SETS = json.loads('''{"frozen": {"dlk": "dlk", "edw": "edw", "dlk_cda": "dlk_bak_20260914", "ods_carry": "ods", "pii": "pii"}, "prod": {"dlk": "prod_dlk", "edw": "prod_edw", "dlk_cda": "prod_dlk", "ods_carry": "ods", "pii": "prod_pii"}}''')
if not hash_salt:
    raise ValueError("hash_salt is empty: set it in ODS_v2_00_Setup (parameters cell)")
if source_set not in SOURCE_SETS:
    raise ValueError(f"source_set must be one of {list(SOURCE_SETS)}")
batch_id = batch_id or datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d%H%M%S")
conf = {"ods.target": target_schema, "ods.stg": stg_schema, "ods.cutoff_date": cutoff_date,
        "ods.batch_id": batch_id, **{f"ods.{k}": v for k, v in SOURCE_SETS[source_set].items()}}
spark.conf.set("spark.sql.variable.substitute", "true")
spark.conf.set("spark.sql.session.timeZone", "UTC")
for k, v in conf.items():
    spark.conf.set(k, v)
spark.sql(f"create schema if not exists {target_schema}")
spark.sql(f"create schema if not exists {stg_schema}")
print(conf)

# Hash functions used by the SQL cells. The salt stays in this closure: it is never written into SQL text or plans.
import hashlib, re
import pandas as pd
from pyspark.sql.functions import pandas_udf
_SALT = hash_salt
_EMAIL = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
def _sha(v):
    return hashlib.sha256((_SALT + v).encode("utf-8")).hexdigest()
def _missing(x):                        # pandas hands an all-null batch over as NaN, not None
    return x is None or (isinstance(x, float) and x != x)
def _email(x):
    if _missing(x): return None
    v = str(x).strip().lower()
    return _sha(v) if _EMAIL.match(v) and v != "gdpr" else None
def _phone(x):
    if _missing(x): return None
    d = re.sub(r"[^0-9]", "", str(x))
    if len(d) < 7: return None
    if d.startswith("00353"): d = "0" + d[5:]
    elif d.startswith("353"): d = "0" + d[3:]
    elif len(d) == 9 and d.startswith("8"): d = "0" + d
    return _sha(d)
_ID_NULLS = {"", "GDPR", "NULL", "NONE", "NAN"}
def _id(x, extra):
    if _missing(x): return None
    v = re.sub(r"[^A-Za-z0-9]", "", str(x)).upper()
    return None if v in _ID_NULLS or (extra and v in extra.split(",")) else _sha(v)
_EMAIL_IN = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
_MOBILE_IN = re.compile(r"(?<![0-9A-Za-z-])(\+?353 ?|0)8[35-9][ -]?[0-9]{3}[ -]?[0-9]{4}(?![0-9A-Za-z-])")
def _redact(x):
    if _missing(x): return None
    return _MOBILE_IN.sub("[phone]", _EMAIL_IN.sub("[email]", str(x)))
def _area(x):
    if _missing(x): return None
    v = re.sub(r"[^A-Za-z0-9]", "", str(x)).upper()
    return v[:3] if len(v) >= 3 else None
@pandas_udf("string")
def ods_hash_email(s: pd.Series) -> pd.Series: return s.map(_email)
@pandas_udf("string")
def ods_hash_phone(s: pd.Series) -> pd.Series: return s.map(_phone)
@pandas_udf("string")
def ods_hash_id(s: pd.Series, extra: pd.Series) -> pd.Series: return pd.Series([_id(a, b) for a, b in zip(s, extra)])
@pandas_udf("string")
def ods_eircode_area(s: pd.Series) -> pd.Series: return s.map(_area)
@pandas_udf("string")
def ods_redact(s: pd.Series) -> pd.Series: return s.map(_redact)
for f in (ods_hash_email, ods_hash_phone, ods_hash_id, ods_eircode_area, ods_redact):
    spark.udf.register(f.__name__, f)

# Run log: one row per notebook start / success / failure, so ODS_v2_Run can resume a batch where it failed.
from pyspark.sql import functions as _F
def ods_run_log(notebook, status, detail=""):
    (spark.createDataFrame([(batch_id, notebook, status, detail)], "OdsBatchId string, Notebook string, Status string, Detail string")
          .withColumn("LoggedAt", _F.current_timestamp())
          .write.mode("append").saveAsTable(f"{target_schema}.OdsRunLog"))


In [ ]:
%%pyspark
ods_run_log("Full", "started")


## Policy


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.Policy                                                                       */
/*  Description : Current state of every policy: the highest RecordVersion per policy code (RecordVersi */
/*              : onIsLatest is known to be                                                             */
/*              : unreliable, so it is carried, not used). ClientCode falls back to the first 6 charact */
/*              : ers of a 9-character code.                                                            */
/*              : EdwPolicyKey is the warehouse key, kept for reconciliation.                           */
/*              :                                                                                       */
/*              : Grain: one row per policy code, its latest version                                    */
/*              : Business key: PolicyCode   Surrogate key: PolicyId                                    */
/*              : History: current (all versions in PolicyVersion)   Cut-off: none                      */
/*              : Source: edw.tbl_ref_policy                                                            */
/*              : Reads: edw.tbl_ref_policy                                                             */
/******************************************************************************************************/
create or replace table ${ods.target}.Policy as
select  row_number() over (order by PolicyCode)                     as PolicyId,
        x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  nullif(upper(trim(p.PolicyCode)), '')                                              as PolicyCode,
                coalesce(nullif(upper(trim(p.ClientCode)), ''),
                         case when length(trim(p.PolicyCode)) = 9 then upper(left(trim(p.PolicyCode), 6)) end)
                                                                                as ClientCode,
                p.PolicyKey                                                     as EdwPolicyKey,
                p.ClientKey                                                     as EdwClientKey,
                p.PolicyTypeCode,
                p.PolicyTypeDesc,
                p.PolicyTypeGroup,
                p.DeptCode,
                p.DeptDesc,
                p.DeptGroup,
                p.InsCoCode,
                p.InsCoName,
                nullif(upper(trim(p.InsPolNumber)), '')                                            as InsPolNumber,
                p.PolicyHandlerCode,
                p.PolicyStatusCode,
                p.PolicyStatusDesc,
                p.PolicySalesDate,
                p.PolicyInceptDate,
                p.PolicyRenewDate,
                p.PolicyLapsedDate,
                p.PolicyCancelDate,
                p.CurrencyCode,
                p.CCYGrossPremium,
                p.CCYCommission,
                p.CCYFees,
                p.CommissionRate,
                p.PaymentType,
                p.FinanceFlag,
                p.RenewOnlineEnabled,
                p.SchemeID,
                p.SchemeName,
                p.SchemeCode,
                p.SchemeProfile,
                p.TelematicsFlag,
                p.EDIFlag,
                p.PromoCodeAmount,
                p.PromoCodeDate,
                p.PromoCodeTreatment,
                p.SegmentCluster,
                p.LYKeyCareFlag,
                p.RecordVersion                                                 as LatestRecordVersion,
                p.RecordVersionIsLatest,
                p.RecordDateTimeStart                                           as LatestRecordDateTimeStart
        from    (
                select  *,
                        row_number() over (partition by upper(trim(PolicyCode)) order by RecordVersion desc) as rn
                from    ${ods.edw}.tbl_ref_policy
                where   nullif(trim(PolicyCode), '') is not null
                ) p
        where   p.rn = 1
        ) x

## PolicyVersion


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyVersion                                                                */
/*  Description : Every version of every policy with its validity interval: status, dates, insurer, sch */
/*              : eme and premium as they                                                               */
/*              : stood. For "what did this policy look like at time t" and status-change history.      */
/*              :                                                                                       */
/*              : Grain: one row per policy code and record version                                     */
/*              : Business key: PolicyCode + RecordVersion   Surrogate key: PolicyVersionId             */
/*              : History: versioned (RecordDateTimeStart / RecordDateTimeEnd)   Cut-off: none          */
/*              : Source: edw.tbl_ref_policy                                                            */
/*              : Reads: edw.tbl_ref_policy, ods_v2.Policy                                              */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyVersion as
select  row_number() over (order by PolicyCode, RecordVersion)      as PolicyVersionId,
        x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  p.PolicyId,
                nullif(upper(trim(v.PolicyCode)), '')                                              as PolicyCode,
                v.RecordVersion,
                v.RecordVersionIsLatest,
                v.RecordDateTimeStart,
                v.RecordDateTimeEnd,
                nullif(upper(trim(v.ClientCode)), '')                                              as ClientCode,
                v.PolicyTypeCode,
                v.PolicyTypeGroup,
                v.InsCoCode,
                v.InsCoName,
                nullif(upper(trim(v.InsPolNumber)), '')                                            as InsPolNumber,
                v.PolicyStatusCode,
                v.PolicyStatusDesc,
                v.PolicySalesDate,
                v.PolicyInceptDate,
                v.PolicyRenewDate,
                v.PolicyLapsedDate,
                v.PolicyCancelDate,
                v.CCYGrossPremium,
                v.CCYCommission,
                v.CCYFees,
                v.PaymentType,
                v.FinanceFlag,
                v.SchemeName,
                v.SchemeCode,
                v.RenewOnlineEnabled
        from    ${ods.edw}.tbl_ref_policy   v
        join    ${ods.target}.Policy        p   on p.PolicyCode = upper(trim(v.PolicyCode))
        ) x

## PolicyTravel


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_travel_policy_link                                                   */
/*  Description : One Relay travel policy per insurer policy number (the lowest PolicyId where several  */
/*              : share it).                                                                            */
/*              :                                                                                       */
/*              : Reads: ods_v2.Policy                                                                  */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_travel_policy_link as
select  InsPolNumber, min(PolicyId) as PolicyId
from    ${ods.target}.Policy
where   PolicyTypeGroup = 'Travel'
and     InsPolNumber is not null
group by InsPolNumber

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyTravel                                                                 */
/*  Description : Travel policy detail from the Mapfre feed (latest UpdatedDate per policy), as noteboo */
/*              : k 01. Joined to the Relay                                                             */
/*              : policy by InsPolNumber = PolicyReference (MAPCHILL...). Name, address, email and phon */
/*              : e are not taken.                                                                      */
/*              :                                                                                       */
/*              : Grain: one row per Mapfre travel policy id, its latest update                         */
/*              : Business key: MapfrePolicyId                                                          */
/*              : History: current   Cut-off: none                                                      */
/*              : Source: dlk.ext_travel_policy_trans + dlk.ext_travel_policy                           */
/*              : Reads: dlk.ext_travel_policy_trans, dlk.ext_travel_policy, stg.odsv2_travel_policy_li */
/*              : nk                                                                                    */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyTravel as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  l.PolicyId,
                t.PolicyId                                                      as MapfrePolicyId,
                concat('M', t.MapfreCustomerId)                                 as ClientCode,
                t.MapfreCustomerId,
                t.QuoteId                                                       as MapfreQuoteId,
                nullif(upper(trim(t.PolicyReference)), '')                                         as InsPolicyCode,
                t.ProductName,
                t.PolicyTypeName,
                t.CreatedDate,
                t.UpdatedDate                                                   as LastUpdatedDate,
                t.PurchaseDate                                                  as SalesDate,
                t.StartDate                                                     as PolicyStartDate,
                t.EndDate                                                       as PolicyEndDate,
                t.TotalPremium,
                t.TravelPremium,
                t.MedicalPremium,
                t.ExpectedFee,
                t.AdminFee,
                t.HazActivityCount,
                t.MedicalCount,
                t.MKT_OPT_IN,
                b.Source                                                        as SalesChannel,
                t.VoidFlag,
                t.VoidDate,
                b.RenewalFlag,
                b.ExpectedCommission,
                b.Promocode
        from    (
                select  *, row_number() over (partition by PolicyId order by UpdatedDate desc) as rn
                from    ${ods.dlk}.ext_travel_policy_trans
                ) t
        left join ${ods.dlk}.ext_travel_policy          b   on b.PolicyId = t.PolicyId
        left join ${ods.stg}.odsv2_travel_policy_link   l   on l.InsPolNumber = upper(trim(t.PolicyReference))
        where   t.rn = 1
        ) x

## PolicyTravelTransaction


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyTravelTransaction                                                      */
/*  Description : Every Mapfre travel policy transaction (new business, renewals, voids, amendments) wi */
/*              : th its record type, not only                                                          */
/*              : the latest per policy (that is PolicyTravel). ClientCode is 'M' + MapfreCustomerId, t */
/*              : he form the SCV link                                                                  */
/*              : (ods.scveadmlinkfinal, system 5) holds. Name, address, email and phone are not taken. */
/*              :                                                                                       */
/*              : Grain: one row per Mapfre travel policy transaction                                   */
/*              : Business key: TravelTransactionKey                                                    */
/*              : History: event   Cut-off: UpdatedDate                                                 */
/*              : Source: dlk.ext_travel_policy_trans                                                   */
/*              : Reads: dlk.ext_travel_policy_trans, stg.odsv2_travel_policy_link                      */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyTravelTransaction as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  t.EXT_Travel_Policy_TransKey                                    as TravelTransactionKey,
                l.PolicyId,
                t.PolicyId                                                      as MapfrePolicyId,
                concat('M', t.MapfreCustomerId)                                 as ClientCode,
                t.MapfreCustomerId,
                t.QuoteId                                                       as MapfreQuoteId,
                t.PreviousQuoteId                                               as MapfrePreviousQuoteId,
                t.RecordType,
                t.Source,
                t.RenewalFlag,
                nullif(upper(trim(t.PolicyReference)), '')                                         as InsPolicyCode,
                t.ProductName,
                t.PolicyTypeName,
                t.CreatedDate,
                t.UpdatedDate,
                t.PurchaseDate,
                t.StartDate,
                t.EndDate,
                t.Promocode,
                t.TotalPremium,
                t.TravelPremium,
                t.MedicalPremium,
                t.ExpectedFee,
                t.AdminFee,
                t.HazActivityCount,
                t.MedicalCount,
                t.MKT_OPT_IN,
                t.VoidFlag,
                t.VoidDate
        from    ${ods.dlk}.ext_travel_policy_trans              t
        left join ${ods.stg}.odsv2_travel_policy_link   l   on l.InsPolNumber = upper(trim(t.PolicyReference))
        where   t.UpdatedDate >= date '${ods.cutoff_date}'
        ) x

## PolicyMotorProposer


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyMotorProposer                                                          */
/*  Description : Motor proposer detail per policy version, carried from the current ods (loaded from t */
/*              : he SQL Server warehouse;                                                              */
/*              : there is no tbl_ref_policy_motor in Fabric). To be rebuilt from dlk.rbsdata2_motor_*  */
/*              : with the DWH logic.                                                                   */
/*              :                                                                                       */
/*              : Grain: one row per policy code and record version                                     */
/*              : Business key: PolicyCode + RecordVersion                                              */
/*              : History: versioned (RecordVersion)   Cut-off: none                                    */
/*              : Source: ods.policymotorproposer (carried forward; no edw source in Fabric)            */
/*              : Reads: ods.policymotorproposer, ods_v2.Policy                                         */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyMotorProposer as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  p.PolicyId,
                nullif(upper(trim(m.PolicyCode)), '')                                              as PolicyCode,
                m.RecordVersion,
                m.ProposerResidentSinceDate,
                m.ProposerLicenceTypeDesc,
                m.ProposerLicenceIssueCountry,
                m.ProposerLicenceIssueDate,
                m.ProposerEmploymentStatusDesc,
                m.ProposerOccupationDesc,
                m.ProposerNCDYears,
                m.ProposerNCDProtection,
                m.ProposerNCDCountry,
                m.ProposerPreviousInsFlag,
                m.ProposerPreviouslyImposedTermsFlag,
                m.ProposerPreviouslyRefusedFlag,
                m.ProposerNonMotoringConvictionsFlag,
                m.ndVehicleFlag                                                 as SecondVehicleFlag,
                m.RiskProfileBand,
                m.ProposerAge,
                m.ProposerNDEYears
        from    (select distinct * from ${ods.ods_carry}.policymotorproposer) m
        left join ${ods.target}.Policy  p   on p.PolicyCode = upper(trim(m.PolicyCode))
        ) x

## PolicyMotorVehicle


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyMotorVehicle                                                           */
/*  Description : Motor vehicle detail per policy version, carried from the current ods. The registrati */
/*              : on is held only as a                                                                  */
/*              : salted hash (VehicleRegKey).                                                          */
/*              :                                                                                       */
/*              : Grain: one row per policy code and record version                                     */
/*              : Business key: PolicyCode + RecordVersion                                              */
/*              : History: versioned (RecordVersion)   Cut-off: none                                    */
/*              : Source: ods.policymotorvehicle (carried forward; no edw source in Fabric)             */
/*              : Reads: ods.policymotorvehicle, ods_v2.Policy                                          */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyMotorVehicle as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  p.PolicyId,
                nullif(upper(trim(v.PolicyCode)), '')                                              as PolicyCode,
                v.RecordVersion,
                v.VehicleManufacturer,
                v.VehicleModel,
                v.VehicleManufacturedYearFrom,
                v.VehicleManufacturedYearTo,
                v.VehicleType,
                v.VehicleFuelType,
                v.VehicleEngineCC,
                v.VehicleImportedFlag,
                v.VehicleAlarmFlag,
                v.VehicleTrailerFlag,
                v.VehicleExcessAmount,
                v.VehiclePurchaseDate,
                v.VehicleValue,
                v.VehicleMileageBusiness,
                v.VehicleMileagePrivate,
                v.VehicleRegIrelandFlag,
                ods_hash_id(v.VehicleRegNumber, '')                                     as VehicleRegKey,
                v.VehicleRegDate,
                v.VehicleCoverType,
                v.VehicleValidDrivers,
                v.VehicleNoOfLiveDrivers,
                v.VehicleClassOfUse,
                v.VehicleMainUseAreaCode,
                v.VehicleMainUseArea,
                v.VehicleMainUseAreaCounty,
                v.VehicleKeptOvernightAreaCode,
                v.VehicleKeptOvernightArea,
                v.VehicleKeptOvernightCounty,
                v.VehicleLHDFlag
        from    (select distinct * from ${ods.ods_carry}.policymotorvehicle) v
        left join ${ods.target}.Policy  p   on p.PolicyCode = upper(trim(v.PolicyCode))
        ) x

## PolicyHome


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyHome                                                                   */
/*  Description : Home risk detail per policy version, carried from the current ods. The Eircode is red */
/*              : uced to its routing key                                                               */
/*              : (HomeEircodeArea); the full code is not held.                                         */
/*              :                                                                                       */
/*              : Grain: one row per policy code and record version                                     */
/*              : Business key: PolicyCode + RecordVersion                                              */
/*              : History: versioned (RecordVersion)   Cut-off: none                                    */
/*              : Source: ods.policyhome (carried forward; no edw source in Fabric)                     */
/*              : Reads: ods.policyhome, ods_v2.Policy                                                  */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyHome as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  p.PolicyId,
                nullif(upper(trim(h.PolicyCode)), '')                                              as PolicyCode,
                h.RecordVersion,
                h.HomeFirstTimeBuyerFlag,
                h.HomeNCDYears,
                h.HomeNCDYearsGranted,
                h.HomeCCRequiredSumInsured,
                h.HomeCCGrantedSumInsured,
                h.HomeCCAccidentalDamageGrantedFlag,
                h.HomeBCRequiredSumInsured,
                h.HomeBCGrantedSumInsured,
                h.HomeBCAccidentalDamageGrantedFlag,
                h.HomeSICNoOfItemsCovered,
                h.HomeSICRequiredSumInsured,
                h.HomeSICGrantedSumInsured,
                h.HomePropertyType,
                h.HomeConstructionYear,
                h.HomeNoOfBathrooms,
                h.HomeNoOfBedrooms,
                h.HomeNoOfSmokeAlarms,
                h.HomeHeatingType,
                h.HomeRoofConstructionType,
                h.HomeRoofNonStandardPercentage,
                h.HomeLockType,
                h.HomeAlarmType,
                h.HomeNeighbourhoodWatchFlag,
                h.HomeProposerType,
                h.HomeResidenceType,
                h.HomeOccupiedDuringDayFlag,
                ods_eircode_area(h.HomeAddressPostCode)                             as HomeEircodeArea
        from    (select distinct * from ${ods.ods_carry}.policyhome) h
        left join ${ods.target}.Policy  p   on p.PolicyCode = upper(trim(h.PolicyCode))
        ) x

## PolicySale


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicySale                                                                   */
/*  Description : Sale postings (new business, renewals, rebooks) for all products. Lead / handler pers */
/*              : on names and the email                                                                */
/*              : address are not taken; the registration is a salted hash.                             */
/*              :                                                                                       */
/*              : Grain: one row per policy code and reporting sale date                                */
/*              : Business key: PolicyCode + ReportingSaleDate                                          */
/*              : History: event (all sales, no cut-off)   Cut-off: none                                */
/*              : Source: edw.tbl_fact_policy_sales                                                     */
/*              : Reads: edw.tbl_fact_policy_sales, ods_v2.Policy                                       */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicySale as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  p.PolicyId,
                nullif(upper(trim(s.PolicyCode)), '')                                              as PolicyCode,
                s.ReportingSaleDate,
                s.EffectiveDate,
                s.WorkDate,
                s.WorkWeek,
                s.ReportingSaleType,
                s.ReportingSaleCategory,
                s.PolicyTypeCode,
                s.PolicyTypeDesc,
                s.PolicyTypeGroup,
                s.PolicyStatusCode,
                s.PolicyStatusDesc,
                s.PolicyCloseNum,
                s.PolicyRenewDateAdj,
                s.PolicyRenewMonth,
                s.PolicyKeyEffDate                                              as EdwPolicyKeyEffDate,
                s.PolicyInceptDate,
                nullif(upper(trim(s.LYPolicyCode)), '')                                            as LYPolicyCode,
                s.LYReportingSaleDate,
                nullif(upper(trim(s.OLDPolicyCode)), '')                                           as OldPolicyCode,
                s.RiskProfileBand,
                s.InsCoCode,
                s.InsCoName,
                ods_redact(nullif(upper(trim(s.InsPolNumber)), ''))                                    as InsPolNumber,
                s.FinanceFlag,
                s.RenewalTransferFlag,
                s.MTARebookedFlag,
                nullif(lower(trim(s.QuoteQueryGUID)), '')                                          as QuoteQueryGuid,
                s.Channel,
                s.SubChannel,
                s.ChannelCall,
                s.ChannelWeb,
                s.ChannelXSell,
                s.ChannelOnLine,
                s.OrgID,
                s.SaleCount,
                s.EURGrossPremium,
                s.EURNetPremium,
                s.EURCommission,
                s.EURFees,
                s.SchemeName,
                s.SchemeCode,
                s.SchemeProfile,
                s.DeptGroup,
                s.CloserEventDate,
                s.DiaryPaymentTypeTY,
                s.DiaryPaymentTypeLY,
                s.RenewalsPortal,
                s.RenewalOnlineEnabled,
                s.VehicleCoverType,
                ods_hash_id(s.VehicleRegNumber, '')                                     as VehicleRegKey,
                s.CampaignKey,
                s.CampaignName,
                s.MFQQuoteSummaryID,
                s.MFQCreatedDatetime,
                s.MFQMatchType,
                s.MFQQuoteType,
                s.HEQReference,
                s.HEQCreatedDateTime,
                s.HEQMatchType,
                s.HEQQuoteType,
                s.kci_flag                                                      as KeyCareFlag,
                s.kci_type                                                      as KeyCareType,
                s.kci_premium                                                   as KeyCarePremium,
                s.kci_commission                                                as KeyCareCommission,
                s.kci_workDate                                                  as KeyCareWorkDate
        from    ${ods.edw}.tbl_fact_policy_sales    s
        left join ${ods.target}.Policy              p   on p.PolicyCode = upper(trim(s.PolicyCode))
        ) x

## PolicyRenewal


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyRenewal                                                                */
/*  Description : One row per renewal cycle of a policy: last-year term, the offer / invite, the outcom */
/*              : e (this-year policy code),                                                            */
/*              : portal logins and payments. PolicyCode is the last-year (expiring) policy; TYPolicyCo */
/*              : de the renewed one.                                                                   */
/*              : Phone numbers are not taken; registrations are salted hashes.                         */
/*              :                                                                                       */
/*              : Grain: one row per policy code and last-year renew date                               */
/*              : Business key: PolicyCode + LYPolicyRenewDateAdj                                       */
/*              : History: event (all renewal cycles, no cut-off)   Cut-off: none                       */
/*              : Source: edw.tbl_fact_policy_renewals_rebuilt (identical copy, until tbl_fact_policy_r */
/*              : enewals is reinstated)                                                                */
/*              : Reads: edw.tbl_fact_policy_renewals_rebuilt, ods_v2.Policy                            */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyRenewal as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  p.PolicyId,
                nullif(upper(trim(r.PolicyCode)), '')                                              as PolicyCode,
                r.LYPolicyRenewDateAdj,
                r.RenewalMonth,
                r.LYEffectiveDate,
                r.LYPolicyTypeCode,
                r.LYPolicyTypeGroup,
                r.LYPolicyStatusDesc,
                r.LYReportingSaleType,
                r.LYReportingSaleCategory,
                r.LYReportingSaleDate,
                r.LYInsCoCode,
                r.LYInsCoName,
                r.LYSchemeName,
                r.LYEURGrossPremium,
                r.LYEURCommission,
                r.LYEURFees,
                ods_hash_id(r.LYVehicleRegNumber, '')                                   as LYVehicleRegKey,
                r.MTAFlag,
                r.RenEURPremInvite,
                r.RenEURPremInviteOrig,
                r.RenEURPremAlternative,
                r.RenEURPremOffer,
                r.RenEURFee,
                r.PolicyOfferNum,
                r.PolicyRetNum,
                r.PolicyRetType,
                r.OfferType,
                nullif(upper(trim(r.NewPolicyCode)), '')                                           as NewPolicyCode,
                r.NewPolicyCodeRule,
                nullif(upper(trim(r.TYPolicyCode)), '')                                            as TYPolicyCode,
                r.TYEffectiveDate,
                r.TYInsCoCode,
                r.TYInsCoName,
                r.TYSchemeName,
                r.TYReportingSaleType,
                r.TYReportingSaleCategory,
                r.TYReportingSaleDate,
                r.TYPolicyRenewDateAdj,
                r.TYPolicyStatusDesc,
                r.TYEURGrossPremium,
                r.TYEURCommission,
                r.TYEURFees,
                ods_hash_id(r.TYVehicleRegNumber, '')                                   as TYVehicleRegKey,
                r.PolicyCodeRevisedAtOffer,
                r.RiskProfileBand,
                r.ClientAgeBand,
                r.VehicleCoverType,
                r.Channel,
                r.SubChannel,
                r.RenewalsPortal,
                r.DiaryPaymentTypeTY,
                r.PromoCodeApplied,
                r.PromoCodeAmount,
                r.PromoCodeDate,
                r.PromoCodeTreatment,
                r.SuccessfulLoginCount,
                r.FailedLoginCount,
                r.SuccessfulPaymentCount,
                r.FailedPaymentCount,
                r.LastCallDateTime,
                r.MFQSale,
                r.MFQSaleDate,
                r.ClientTenure,
                r.ProductTenure,
                r.CreateDate                                                    as EdwCreateDate
        from    ${ods.edw}.tbl_fact_policy_renewals_rebuilt r
        left join ${ods.target}.Policy              p   on p.PolicyCode = upper(trim(r.PolicyCode))
        ) x

## PolicyMonthEnd


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyMonthEnd                                                               */
/*  Description : Month-end position of each policy: open / close counts, movements in the month, close */
/*              :  premium, status and the                                                              */
/*              : holder's banded attributes as at that month-end. Date of birth and postcode are not t */
/*              : aken.                                                                                 */
/*              :                                                                                       */
/*              : Grain: one row per policy code and month-end date                                     */
/*              : Business key: PolicyCode + EffectiveDate                                              */
/*              : History: snapshot (month-end)   Cut-off: EffectiveDate                                */
/*              : Source: edw.tbl_fact_policy_mvt                                                       */
/*              : Reads: edw.tbl_fact_policy_mvt, ods_v2.Policy                                         */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyMonthEnd as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  p.PolicyId,
                nullif(upper(trim(m.PolicyCode)), '')                                              as PolicyCode,
                m.EffectiveDate,
                nullif(upper(trim(m.ClientCode)), '')                                              as ClientCode,
                m.OpenNum,
                m.CloseNum,
                m.NewBusGrossNum,
                m.RenewGrossNum,
                m.MTAGrossNum,
                m.MTCGrossNum,
                m.DueRenewalNum,
                m.OpenCloseDesc,
                m.EURGrossPremClose,
                m.EURCommissionClose,
                m.EURFeesClose,
                m.PolicyStatusCode,
                m.PolicyStatusDesc,
                m.PolicyTypeGroup,
                m.ReportingSaleType,
                m.ReportingSaleCategory,
                m.ReportingSaleDate,
                m.PolicyInceptDate,
                m.PolicyRenewDate,
                m.PolicyRenewDateAdj,
                m.PolicyLapsedDate,
                m.PolicyLapsedFlag,
                m.PolicyCancelDate,
                m.PolicyCancelFlag,
                m.InsCoCode,
                m.InsCoName,
                m.FinanceFlag,
                m.ClientAge,
                m.ClientAgeBand,
                m.ClientGender,
                m.ClientCounty,
                m.ClientProvince,
                ods_hash_id(m.VehicleRegNumber, '')                                     as VehicleRegKey,
                m.VehicleManufacturer,
                m.VehicleModel,
                m.VehicleCoverType,
                m.VehicleFuelType,
                m.RiskProfileBand,
                m.ClientTenure,
                m.ProductTenure,
                m.InsurerTenure,
                m.ChannelGroup,
                m.DiaryPaymentTypeTY
        from    ${ods.edw}.tbl_fact_policy_mvt  m
        left join ${ods.target}.Policy          p   on p.PolicyCode = upper(trim(m.PolicyCode))
        where   m.EffectiveDate >= date '${ods.cutoff_date}'
        ) x

## PolicyCancellation


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_policy_cancellation                                                  */
/*  Description : Cancellation rows with a generated id.                                                */
/*              :                                                                                       */
/*              : Reads: edw.tbl_fact_policy_mtc                                                        */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_policy_cancellation as
select  row_number() over (order by c.PolicyCode, c.EffectiveDate, c.ExtractDatetime, c.rn, c.TrCode, c.DrCode,
                                    c.ShortDescription)                         as PolicyCancellationId,
        c.*
from    ${ods.edw}.tbl_fact_policy_mtc c

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyCancellation                                                           */
/*  Description : Mid-term cancellations with reason, diary flags and the money reversed. The source ha */
/*              : s no small natural key, so                                                            */
/*              : each row gets a generated id (ordered by policy, effective date, extract time, row).  */
/*              :                                                                                       */
/*              : Grain: one row per cancellation record (no natural key; generated key)                */
/*              : Business key: PolicyCancellationId                                                    */
/*              : History: event (no cut-off; small)   Cut-off: none                                    */
/*              : Source: edw.tbl_fact_policy_mtc                                                       */
/*              : Reads: stg.odsv2_policy_cancellation, ods_v2.Policy                                   */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyCancellation as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  c.PolicyCancellationId,
                p.PolicyId,
                nullif(upper(trim(c.PolicyCode)), '')                                              as PolicyCode,
                c.EffectiveDate,
                nullif(upper(trim(c.ClientCode)), '')                                              as ClientCode,
                c.PolicyTypeGroup,
                c.Channel,
                c.InsCoCode,
                c.InsCoName,
                c.ReportingSaleDate,
                c.ReportingSaleCategory,
                c.PolicyStatusCode,
                c.PolicyStatusDesc,
                c.PolicyInceptionDate,
                c.DaysOnCover,
                c.PolicyRenewDate,
                c.PolicyCancelDate,
                c.PolicyCancelEffDate,
                c.PolicyMTCDate,
                c.Rebooked,
                c.CXLPolicy,
                nullif(upper(trim(c.OldPolicyCode)), '')                                           as OldPolicyCode,
                c.OldPolicyStatusDesc,
                c.DiaryPaymentTypeTY,
                c.OriginalGWP,
                c.OriginalFee,
                c.CancelledGWP,
                c.OriginalIPP,
                c.CancelledIPP,
                c.DrCode,
                c.TrCode,
                c.PlCancelReason,
                c.Category,
                c.ShortDescription,
                c.FullDescription,
                c.CancellationDiaryFlag,
                c.CPFCAncellationArrearsFlag                                    as CPFCancellationArrearsFlag,
                c.EurBalance,
                c.MTCNetNum,
                c.EURCommissionMTC,
                c.EURFeesMTC,
                c.ExtractDatetime,
                c.WorkDate
        from    ${ods.stg}.odsv2_policy_cancellation c
        left join ${ods.target}.Policy              p   on p.PolicyCode = upper(trim(c.PolicyCode))
        ) x

## PolicyTransaction


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyTransaction                                                            */
/*  Description : Relay accounts archive (premium, fee, commission postings) with the premium type from */
/*              :  the warehouse mapping                                                                */
/*              : (joined with spaces in the Relay codes read as '-', as the warehouse and notebook 04  */
/*              : do).                                                                                  */
/*              : Relay yyyymmdd decimals are turned into dates. Exact duplicate rows in the source (ab */
/*              : out 80k) are dropped.                                                                 */
/*              :                                                                                       */
/*              : Grain: one row per Relay accounts transaction (exact duplicate rows removed)          */
/*              : Business key: PolicyCode + PostingDate + TransSeq + ArCode + ArNumber + ArReference + */
/*              :  ClientTransType + ClientAmount   Surrogate key: PolicyTransactionId                  */
/*              : History: event   Cut-off: PostingDate                                                 */
/*              : Source: dlk.rbsdata2_newarchive + edw.tbl_ref_mapping_bar_premium_type                */
/*              : Reads: dlk.rbsdata2_newarchive, edw.tbl_ref_mapping_bar_premium_type                  */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyTransaction as
select  row_number() over (order by PolicyCode, PostingDate, TransSeq, ArCode, ArNumber, ArReference, ClientTransType, ClientAmount)as PolicyTransactionId,
        x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  distinct
                nullif(upper(trim(a.ar_policy_code)), '')                                          as PolicyCode,
                nullif(upper(trim(a.ar_client_code)), '')                                          as ClientCode,
                case when a.ar_posting_date > 19000000 then to_date(cast(cast(a.ar_posting_date as bigint) as string), 'yyyyMMdd') end                                   as PostingDate,
                case when a.ar_effective_date > 19000000 then to_date(cast(cast(a.ar_effective_date as bigint) as string), 'yyyyMMdd') end                                 as EffectiveDate,
                a.ar_trans_seq                                                  as TransSeq,
                a.ar_code                                                       as ArCode,
                a.ar_number                                                     as ArNumber,
                ods_redact(a.ar_reference)                                          as ArReference,
                a.ar_cli_type                                                   as ClientTransType,
                a.ar_cli_amount                                                 as ClientAmount,
                a.ar_cli_net_premium                                            as ClientNetPremium,
                a.ar_cli_commission                                             as ClientCommission,
                a.ar_cli_fees                                                   as ClientFees,
                a.ar_cli_discount                                               as ClientDiscount,
                a.ar_annual_prem                                                as AnnualPremium,
                a.ar_com_type                                                   as InsurerTransType,
                a.ar_com_sta                                                    as InsurerStatus,
                a.ar_com_initial_net_prem                                       as InsurerInitialNetPremium,
                a.ar_com_initial_comm                                           as InsurerInitialCommission,
                a.ar_confirmed_prov                                             as ConfirmedProvisional,
                a.ar_return_sub_type                                            as ReturnSubType,
                a.ar_insco                                                      as InsCoCode,
                a.ar_pol_type                                                   as PolicyType,
                a.ar_new_flag                                                   as NewFlag,
                a.ar_show_on_report                                             as ShowOnReport,
                a.ar_dtype                                                      as DType,
                a.ar_trans_description                                          as TransDescription,
                a.ar_addon_ref                                                  as AddonRef,
                a.ar_department                                                 as Department,
                a.ar_handler                                                    as HandlerCode,
                a.ar_user                                                       as UserCode,
                a.ar_paymenttype                                                as PaymentType,
                a.ar_paymentmethod                                              as PaymentMethod,
                a.ar_lapse_reason_type                                          as LapseReasonType,
                a.ar_lapse_reason_code                                          as LapseReasonCode,
                m.PremiumType,
                m.PremiumTypeGroup
        from    ${ods.dlk}.rbsdata2_newarchive                  a
        left join ${ods.edw}.tbl_ref_mapping_bar_premium_type   m          -- Relay codes hold spaces where the mapping has '-'
                on  m.ar_cli_type        = replace(a.ar_cli_type, ' ', '-')
                and m.ar_com_type        = replace(a.ar_com_type, ' ', '-')
                and m.ar_return_sub_type = replace(a.ar_return_sub_type, ' ', '-')
                and m.ar_com_sta         = replace(a.ar_com_sta, ' ', '-')
                and m.ar_confirmed_prov  = replace(a.ar_confirmed_prov, ' ', '-')
        where   case when a.ar_posting_date > 19000000 then to_date(cast(cast(a.ar_posting_date as bigint) as string), 'yyyyMMdd') end >= date '${ods.cutoff_date}'
        ) x

## PolicyMta


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyMta                                                                    */
/*  Description : Relay mid-term adjustment history: what changed (the Change_* / Amend_* flags), premi */
/*              : um and fee, effective                                                                 */
/*              : window. Free-text notes and snapshots are not taken (they can hold personal detail).  */
/*              :                                                                                       */
/*              : Grain: one row per policy, history sequence and history type                          */
/*              : Business key: PolicyCode + HistorySeq + HistoryType                                   */
/*              : History: event   Cut-off: ProcessedDate                                               */
/*              : Source: dlk.rbsdata2_mta_history                                                      */
/*              : Reads: dlk.rbsdata2_mta_history                                                       */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyMta as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  nullif(upper(trim(h.Hi_pol_code)), '')                                             as PolicyCode,
                h.Hi_seq                                                        as HistorySeq,
                h.Hi_type                                                       as HistoryType,
                h.hi_sub_type                                                   as HistorySubType,
                h.Hi_trcode                                                     as TrCode,
                case when h.Hi_date > 19000000 then to_date(cast(cast(h.Hi_date as bigint) as string), 'yyyyMMdd') end                                           as HistoryDate,
                h.Processed_Date                                                as ProcessedDate,
                h.Effective_Start_Date                                          as EffectiveStartDate,
                h.Effective_End_Date                                            as EffectiveEndDate,
                h.MTA_Type                                                      as MtaType,
                h.Sub_Type                                                      as MtaSubType,
                h.Premium,
                h.Fee,
                h.Hi_amount                                                     as Amount,
                h.Hi_an_prem                                                    as AnnualPremium,
                h.Hi_Manual                                                     as ManualFlag,
                h.Change_of_Policyholder_Details                                as ChangeOfPolicyholderDetails,
                h.Change_Address                                                as ChangeAddress,
                h.Amend_Licence                                                 as AmendLicence,
                h.Amend_Convictions_Proposer                                    as AmendConvictionsProposer,
                h.Change_NCD_Protection                                         as ChangeNcdProtection,
                h.Change_Policy_Excess                                          as ChangePolicyExcess,
                h.Permanent_Substitution                                        as PermanentSubstitution,
                h.Change_of_Cover                                               as ChangeOfCover,
                h.Change_Class_of_Use                                           as ChangeClassOfUse,
                h.Change_Registration_No                                        as ChangeRegistrationFlag,
                h.Change_Driving_Option                                         as ChangeDrivingOption,
                h.Change_Area_of_Use                                            as ChangeAreaOfUse,
                h.Change_NCD                                                    as ChangeNcd,
                h.Amend_Additional_Items                                        as AmendAdditionalItems,
                h.Amend_Vehicle_Details                                         as AmendVehicleDetails,
                h.Permanent_Additional_Drivers                                  as PermanentAdditionalDrivers,
                h.Amend_Licence_Details                                         as AmendLicenceDetails,
                h.Amend_Convictions_Other                                       as AmendConvictionsOther,
                h.Change_of_Additional_Driver_Details                           as ChangeOfAdditionalDriverDetails,
                h.Amend_2nd_Car_Discount                                        as Amend2ndCarDiscount
        from    ${ods.dlk}.rbsdata2_mta_history h
        where   h.Processed_Date >= date '${ods.cutoff_date}'
        ) x

## PolicyClaimMotor


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_motor_claim_slots                                                    */
/*  Description : The five claim slots as rows, with the extract time used to keep the latest.          */
/*              :                                                                                       */
/*              : Reads: dlk.rbsdata2_motor_clm                                                         */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_motor_claim_slots as
select va_clm_plcode2 as PolicyCode, 1 as ClaimSlot, va_clm_date_1 as ClaimDate, va_clm_type_1 as ClaimType, va_clm_value_1 as ClaimValue, va_clm_fault_1 as OwnFaultFlag, va_clm_inj_1 as InjuryFlag, va_clm_ncb_affected_1 as NcbAffected, va_clm_settled_1 as Settled, coalesce(`$ExtractDateTime`, ReplicationDateTime) as ExtractDateTime from ${ods.dlk}.rbsdata2_motor_clm where va_clm_date_1 > 0
union all
select va_clm_plcode2, 2, va_clm_date_2, va_clm_type_2, va_clm_value_2, va_clm_fault_2, va_clm_inj_2, va_clm_ncb_affected_2, va_clm_settled_2, coalesce(`$ExtractDateTime`, ReplicationDateTime) from ${ods.dlk}.rbsdata2_motor_clm where va_clm_date_2 > 0
union all
select va_clm_plcode2, 3, va_clm_date_3, va_clm_type_3, va_clm_value_3, va_clm_fault_3, va_clm_inj_3, va_clm_ncb_affected_3, va_clm_settled_3, coalesce(`$ExtractDateTime`, ReplicationDateTime) from ${ods.dlk}.rbsdata2_motor_clm where va_clm_date_3 > 0
union all
select va_clm_plcode2, 4, va_clm_date_4, va_clm_type_4, va_clm_value_4, va_clm_fault_4, va_clm_inj_4, va_clm_ncb_affected_4, va_clm_settled_4, coalesce(`$ExtractDateTime`, ReplicationDateTime) from ${ods.dlk}.rbsdata2_motor_clm where va_clm_date_4 > 0
union all
select va_clm_plcode2, 5, va_clm_date_5, va_clm_type_5, va_clm_value_5, va_clm_fault_5, va_clm_inj_5, va_clm_ncb_affected_5, va_clm_settled_5, coalesce(`$ExtractDateTime`, ReplicationDateTime) from ${ods.dlk}.rbsdata2_motor_clm where va_clm_date_5 > 0

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyClaimMotor                                                             */
/*  Description : Motor claims declared on Relay policies: the five claim slots per record turned into  */
/*              : rows (as notebook 04),                                                                */
/*              : latest extract per policy and slot. ClaimDate is null where Relay holds an impossible */
/*              :  date (e.g. 30 Feb);                                                                  */
/*              : ClaimDateRaw keeps the yyyymmdd value as held and is the key.                         */
/*              :                                                                                       */
/*              : Grain: one row per policy, claim date, claim type and value                           */
/*              : Business key: PolicyCode + ClaimDateRaw + ClaimType + ClaimValue + ClaimSlot          */
/*              : History: event (no cut-off; claims history is a risk attribute)   Cut-off: none       */
/*              : Source: dlk.rbsdata2_motor_clm                                                        */
/*              : Reads: stg.odsv2_motor_claim_slots, ods_v2.Policy                                     */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyClaimMotor as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  p.PolicyId,
                nullif(upper(trim(c.PolicyCode)), '')                                              as PolicyCode,
                c.ClaimSlot,
                case when c.ClaimDate > 19000000 then to_date(cast(cast(c.ClaimDate as bigint) as string), 'yyyyMMdd') end                                         as ClaimDate,
                c.ClaimDate                                                     as ClaimDateRaw,
                c.ClaimType,
                c.ClaimValue,
                c.OwnFaultFlag,
                c.InjuryFlag,
                c.NcbAffected,
                c.Settled,
                c.ExtractDateTime
        from    (
                select  *,
                        row_number() over (partition by upper(trim(PolicyCode)), ClaimDate, ClaimType, ClaimValue, ClaimSlot
                                           order by ExtractDateTime desc) as rn
                from    ${ods.stg}.odsv2_motor_claim_slots
                ) c
        left join ${ods.target}.Policy  p   on p.PolicyCode = upper(trim(c.PolicyCode))
        where   c.rn = 1
        ) x

## PolicyClaimHome


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_home_risk_policy                                                     */
/*  Description : One policy code per home risk id (the latest transaction).                            */
/*              :                                                                                       */
/*              : Reads: dlk.ext_home_qs_policydetails                                                  */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_home_risk_policy as
select  upper(RiskId) as HomeRiskId, PolicyCode
from    (
        select  RiskId, PolicyCode,
                row_number() over (partition by upper(RiskId) order by TransactionDate desc) as rn
        from    ${ods.dlk}.ext_home_qs_policydetails
        where   nullif(trim(PolicyCode), '') is not null
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyClaimHome                                                              */
/*  Description : Home claims declared in the home quotation system, linked to the policy code through  */
/*              : the risk id.                                                                          */
/*              :                                                                                       */
/*              : Grain: one row per home claim id                                                      */
/*              : Business key: HomeClaimId                                                             */
/*              : History: event (no cut-off)   Cut-off: none                                           */
/*              : Source: dlk.ext_home_qs_claimdetails + dlk.ext_home_qs_policydetails                  */
/*              : Reads: dlk.ext_home_qs_claimdetails, stg.odsv2_home_risk_policy, ods_v2.Policy        */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyClaimHome as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  c.HomeClaimId,
                p.PolicyId,
                nullif(upper(trim(r.PolicyCode)), '')                                              as PolicyCode,
                upper(c.HomeRiskId)                                             as HomeRiskId,
                c.HomeClaimType,
                c.OccurredAt,
                c.HomeClaimAmount,
                c.Settled,
                c.HomeClaimCircumstance
        from    (
                select  *, row_number() over (partition by HomeClaimId order by `$ExtractDateTime` desc) as rn
                from    ${ods.dlk}.ext_home_qs_claimdetails
                ) c
        left join ${ods.stg}.odsv2_home_risk_policy r   on r.HomeRiskId = upper(c.HomeRiskId)
        left join ${ods.target}.Policy              p   on p.PolicyCode = upper(trim(r.PolicyCode))
        where   c.rn = 1
        ) x

## ClientPortfolio


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.ClientPortfolio                                                              */
/*  Description : Relay portfolio (client) to policy code, as notebook 05 (PolicyCode = portfolio code  */
/*              : + 3-digit sequence).                                                                  */
/*              :                                                                                       */
/*              : Grain: one row per Relay portfolio and policy code                                    */
/*              : Business key: PortfolioId + PolicyCode                                                */
/*              : History: current   Cut-off: none                                                      */
/*              : Source: dlk.relay_policy + dlk.relay_portfolio                                        */
/*              : Reads: dlk.relay_policy, dlk.relay_portfolio, ods_v2.Policy                           */
/******************************************************************************************************/
create or replace table ${ods.target}.ClientPortfolio as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  e.PortfolioId,
                nullif(upper(trim(e.Code)), '')                                                    as ClientCode,
                concat(upper(trim(e.Code)), lpad(cast(d.PolicySequence as string), 3, '0'))
                                                                                as PolicyCode,
                max(p.PolicyId)                                                 as PolicyId
        from    ${ods.dlk}.relay_policy     d
        join    ${ods.dlk}.relay_portfolio  e   on e.PortfolioId = d.PortfolioId
        left join ${ods.target}.Policy      p   on p.PolicyCode = concat(upper(trim(e.Code)), lpad(cast(d.PolicySequence as string), 3, '0'))
        group by e.PortfolioId, upper(trim(e.Code)), concat(upper(trim(e.Code)), lpad(cast(d.PolicySequence as string), 3, '0'))
        ) x

## PolicyDocumentEvent


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_relay_document_event                                                 */
/*  Description : Relay events with their document, business process and event type name.               */
/*              :                                                                                       */
/*              : Reads: dlk.relay_eventmodel_event, dlk.relay_eventmodel_documentevent, dlk.relay_docu */
/*              : mentation_document, dlk.relay_reporting_relaycodelist                                 */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_relay_document_event as
select  distinct
        a.EventId,
        c.DocumentId,
        b.GeneratedByBusinessProcess                                as BusinessProcessCode,
        case b.GeneratedByBusinessProcess
             when 0 then 'Unknown' when 1 then 'New Business' when 2 then 'Permanent MTA'
             when 3 then 'Temp Add Driver' when 4 then 'Cancel Temp Add Driver'
             when 5 then 'Temporary Additional Vehicle' when 6 then 'Cancel Temporary Additional Vehicle'
             when 7 then 'Temp Substatution' when 8 then 'Cancel Temp Substatution' when 9 then 'Suspension'
             when 10 then 'Cancel Suspension' when 11 then 'Cancellation' when 12 then 'Reinstate'
             when 13 then 'Duplicate Certificate' when 14 then 'Renewal Offer' when 15 then 'Renewal Confirmation'
             when 16 then 'Lapse' when 17 then 'Insurer Led Cancelation' when 20 then 'Renewal Transfer'
             when 23 then 'Renewal Chase' when 24 then 'Document Chase' when 25 then 'Prior Year Quotes'
             when 26 then 'Accounts Business Process' when 27 then 'Amend NCD'
             else 'Unknown Business Process' end                    as BusinessProcess,
        a.EventType                                                 as EventTypeCode,
        d.DisplayName                                               as EventType,
        b.Description                                               as DocumentDescription,
        a.CreatedDate                                               as EventDate,
        c.Event_CreatedDate                                         as DocumentEventDate
from    ${ods.dlk}.relay_eventmodel_event           a
join    ${ods.dlk}.relay_eventmodel_documentevent   c   on c.EventId = a.EventId
join    ${ods.dlk}.relay_documentation_document     b   on b.DocumentId = c.DocumentId
left join (
        select  Code, DisplayName
        from    ${ods.dlk}.relay_reporting_relaycodelist
        where   DeletedDate is null and Category = 'Relay.EventModel.EventType'
        ) d     on d.Code = a.EventType
where   a.CreatedDate >= date '${ods.cutoff_date}'

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyDocumentEvent                                                          */
/*  Description : Relay document events (offers, chases, confirmations, cancellations ...) on policies, */
/*              :  as notebook 05: event ->                                                             */
/*              : document -> policy document -> policy event -> policy -> portfolio. Business process  */
/*              : and event type are kept as                                                            */
/*              : code and description rather than one concatenated string. Business process names are  */
/*              : kept exactly as Chill's                                                               */
/*              : code has them, spelling included ('Temp Substatution', 'Insurer Led Cancelation'): co */
/*              : nsumers match on them.                                                                */
/*              :                                                                                       */
/*              : Grain: one row per Relay document event and policy                                    */
/*              : Business key: EventId + DocumentId + PolicyCode                                       */
/*              : History: event   Cut-off: EventDate                                                   */
/*              : Source: dlk.relay_eventmodel_* + dlk.relay_documentation_* + dlk.relay_policy / portf */
/*              : olio                                                                                  */
/*              : Reads: dlk.relay_documentation_policydocument, stg.odsv2_relay_document_event, dlk.re */
/*              : lay_eventmodel_policyevent, dlk.relay_policy, dlk.relay_portfolio, ods_v2.Policy      */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyDocumentEvent as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  distinct
                b.EventId,
                b.DocumentId,
                concat(upper(trim(e.Code)), lpad(cast(d.PolicySequence as string), 3, '0'))
                                                                                as PolicyCode,
                p.PolicyId,
                nullif(upper(trim(e.Code)), '')                                                    as ClientCode,
                d.PolicyId                                                      as RelayPolicyId,
                d.PolicySequence,
                b.BusinessProcessCode,
                b.BusinessProcess,
                b.EventTypeCode,
                b.EventType,
                b.DocumentDescription,
                b.EventDate,
                b.DocumentEventDate
        from    ${ods.dlk}.relay_documentation_policydocument   a
        join    ${ods.stg}.odsv2_relay_document_event           b   on b.DocumentId = a.DocumentId
        join    ${ods.dlk}.relay_eventmodel_policyevent         c   on c.EventId = upper(a.EventId)
        join    ${ods.dlk}.relay_policy                         d   on d.PolicyId = c.PolicyId
        join    ${ods.dlk}.relay_portfolio                      e   on e.PortfolioId = d.PortfolioId
        left join ${ods.target}.Policy                          p
                on  p.PolicyCode = concat(upper(trim(e.Code)), lpad(cast(d.PolicySequence as string), 3, '0'))
        ) x

## RenewalPortalEvent


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.RenewalPortalEvent                                                           */
/*  Description : Applied Renewals portal events: logins (successful / failed) and renewal payments (su */
/*              : ccessful / failed).                                                                   */
/*              :                                                                                       */
/*              : Grain: one row per portal event type and event id                                     */
/*              : Business key: EventType + EventId                                                     */
/*              : History: event   Cut-off: EventDateTime                                               */
/*              : Source: dlk.appliedrenewals_paymentsuccess / paymentfailedevent / successfullogineven */
/*              : t / failedloginevent                                                                  */
/*              : Reads: dlk.appliedrenewals_paymentsuccess, dlk.appliedrenewals_paymentfailedevent, dl */
/*              : k.appliedrenewals_successfulloginevent, dlk.appliedrenewals_failedloginevent          */
/******************************************************************************************************/
create or replace table ${ods.target}.RenewalPortalEvent as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select 'PaymentSuccess' as EventType, Id as EventId, SessionId, `Timestamp` as EventDateTime, nullif(upper(trim(PortfolioCode)), '') as PortfolioCode, nullif(upper(trim(PolicyCodeForRenewal)), '') as PolicyCode, PaymentAmount, PaymentType, cast(null as string) as PaymentErrorCode, cast(null as string) as PaymentErrorMessage, cast(null as boolean) as IsLocked from ${ods.dlk}.appliedrenewals_paymentsuccess where `Timestamp` >= date '${ods.cutoff_date}'
        union all
        select 'PaymentFailed', Id, SessionId, `Timestamp`, nullif(upper(trim(PortfolioCode)), ''), nullif(upper(trim(PolicyCodeForRenewal)), ''), PaymentAmount, PaymentType, PaymentErrorCode, PaymentErrorMessage, cast(null as boolean) from ${ods.dlk}.appliedrenewals_paymentfailedevent where `Timestamp` >= date '${ods.cutoff_date}'
        union all
        select 'LoginSuccess', Id, SessionId, `Timestamp`, nullif(upper(trim(PortfolioCode)), ''), cast(null as string), cast(null as decimal(18,2)), cast(null as string), cast(null as string), cast(null as string), cast(null as boolean) from ${ods.dlk}.appliedrenewals_successfulloginevent where `Timestamp` >= date '${ods.cutoff_date}'
        union all
        select 'LoginFailed', Id, SessionId, `Timestamp`, nullif(upper(trim(PortfolioCode)), ''), cast(null as string), cast(null as decimal(18,2)), cast(null as string), cast(null as string), cast(null as string), isLocked from ${ods.dlk}.appliedrenewals_failedloginevent where `Timestamp` >= date '${ods.cutoff_date}'
        ) x

## DocumentWorkflowStatus


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.DocumentWorkflowStatus                                                       */
/*  Description : MyChill document workflow (document requested / accepted / rejected per policy). Agen */
/*              : t and assigned-to emails are                                                          */
/*              : not taken; the customer's email is a salted hash.                                     */
/*              :                                                                                       */
/*              : Grain: one row per MyChill workflow record (pk + rk)                                  */
/*              : Business key: Pk + Rk                                                                 */
/*              : History: event (no cut-off; small)   Cut-off: none                                    */
/*              : Source: dlk.mychillworkflow_documentstatus                                            */
/*              : Reads: dlk.mychillworkflow_documentstatus                                             */
/******************************************************************************************************/
create or replace table ${ods.target}.DocumentWorkflowStatus as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  pk                                                              as Pk,
                rk                                                              as Rk,
                GUID                                                            as WorkflowGuid,
                nullif(upper(trim(PolicyCode)), '')                                                as PolicyCode,
                DiaryTypeID,
                Title,
                Attempts,
                IsAccepted,
                IsRejected,
                ods_redact(Reasons)                                                 as Reasons,
                AssignedDate,
                `Timestamp`                                                     as EventTimestamp,
                InsertDateTime,
                ods_hash_email(CustomerEmail)                                       as CustomerEmailKey
        from    ${ods.dlk}.mychillworkflow_documentstatus
        ) x

## DocumentUpload


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.DocumentUpload                                                               */
/*  Description : MyChill document uploads per policy. DocumentValue (the value keyed on the document,  */
/*              : e.g. a licence or PPS                                                                 */
/*              : number) is not taken.                                                                 */
/*              :                                                                                       */
/*              : Grain: one row per MyChill upload record (pk + rk)                                    */
/*              : Business key: Pk + Rk                                                                 */
/*              : History: event (no cut-off; small)   Cut-off: none                                    */
/*              : Source: dlk.mychill_newuploaddocumentevents                                           */
/*              : Reads: dlk.mychill_newuploaddocumentevents                                            */
/******************************************************************************************************/
create or replace table ${ods.target}.DocumentUpload as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  pk                                                              as Pk,
                rk                                                              as Rk,
                Id                                                              as UploadId,
                nullif(upper(trim(PolicyCode)), '')                                                as PolicyCode,
                Title,
                DiaryCode,
                Type                                                            as DocumentType,
                PagesUploaded,
                IsUploaded,
                ErrorMEssage                                                    as ErrorMessage,
                `Timestamp`                                                     as EventTimestamp,
                InsertDateTime
        from    ${ods.dlk}.mychill_newuploaddocumentevents
        ) x

## DocumentChaseHome


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.DocumentChaseHome                                                            */
/*  Description : Daily MyChill document chase snapshot for home policies: each required document's dat */
/*              : e and status, and the chase                                                           */
/*              : send dates. Names and mobile number are not taken.                                    */
/*              :                                                                                       */
/*              : Grain: one row per file date and policy code                                          */
/*              : Business key: FileDate + PolicyCode                                                   */
/*              : History: snapshot (daily)   Cut-off: CreateDate                                       */
/*              : Source: edw.exp_mychill_chase_daily_snapshot_home                                     */
/*              : Reads: edw.exp_mychill_chase_daily_snapshot_home                                      */
/******************************************************************************************************/
create or replace table ${ods.target}.DocumentChaseHome as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  FileDate, nullif(upper(trim(PolicyCode)), '') as PolicyCode, PolicyStatus, PolicyTypeGroup, PolicyType, SaleDate, InceptionDate,
                Comp_DDM_Date, Comp_DDM_Status, Gap_In_Cov_Ltr_Date, Gap_In_Cov_Ltr_Status,
                Val_For_Spec_Item_Date, Val_For_Spec_Item_Status, PPS_Num_Date, PPS_Num_Status,
                Identification_Date, Identification_Status, Digital_Journey_Date, Digital_Journey_Status,
                Finance_Form_Date, Finance_Form_Status, Campaign, Day1SendDate, Day7SendDate, Day14SendDate, Day20SendDate,
                CreateDate
        from    ${ods.edw}.exp_mychill_chase_daily_snapshot_home
        where   CreateDate >= date '${ods.cutoff_date}'
        ) x

## DocumentChaseMotorVan


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.DocumentChaseMotorVan                                                        */
/*  Description : Daily MyChill document chase snapshot for motor and van policies (the source repeats  */
/*              : a few file-date + policy                                                              */
/*              : rows: latest kept). Names and mobile number are not taken.                            */
/*              :                                                                                       */
/*              : Grain: one row per file date and policy code                                          */
/*              : Business key: FileDate + PolicyCode                                                   */
/*              : History: snapshot (daily)   Cut-off: CreateDate                                       */
/*              : Source: edw.exp_mychill_chase_daily_snapshot_motorvan                                 */
/*              : Reads: edw.exp_mychill_chase_daily_snapshot_motorvan                                  */
/******************************************************************************************************/
create or replace table ${ods.target}.DocumentChaseMotorVan as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  FileDate, PolicyCodeClean as PolicyCode, PolicyStatus, PolicyTypeGroup, PolicyType, SaleDate, InceptionDate,
                Sign_Prop_Date, Sign_Prop_Status, Sign_Stat_Of_Fact_Date, Sign_Stat_Of_Fact_Status,
                Comp_Fin_Agree_Date, Comp_Fin_Agree_Status, Comp_DDM_Date, Comp_DDM_Status,
                Proof_Of_NCB_Date, Proof_Of_NCB_Status, Trans_Proof_Of_NCB_Date, Trans_Proof_Of_NCB_Status,
                Ltr_Of_Driv_Exp_Date, Ltr_Of_Driv_Exp_Status, Prop_Driv_Lic_Date, Prop_Driv_Lic_Status,
                Nam_Driv_Lic_Date, Nam_Driv_Lic_Status, Gap_In_Cov_Ltr_Date, Gap_In_Cov_Ltr_Status,
                `2ndCar_Cert_Date` as SecondCar_Cert_Date, `2ndCar_Cert_Status` as SecondCar_Cert_Status,
                Doc_Ltr_Date, Doc_Ltr_Status, Engineers_Rpt_Date, Engineers_Rpt_Status, NCT_Date, NCT_Status,
                Veh_Lic_Cert_Date, Veh_Lic_Cert_Status, Irish_Reg_Date, Irish_Reg_Status,
                Main_Driver_Dec_Date, Main_Driver_Dec_Status, Soc_Dom_Pleas_Dec_Date, Soc_Dom_Pleas_Dec_Status,
                Comp_Car_Exper_Date, Comp_Car_Exper_Status, DD_Conf_Ltr_Date, DD_Conf_Ltr_Status,
                Comp_Fin_Mand_Date, Comp_Fin_Mand_Status, Val_For_Spec_Item_Date, Val_For_Spec_Item_Status,
                Dri_Lic_Num_Date, Dri_Lic_Num_Status, Comp_Lost_Cert_Dec_Date, Comp_Lost_Cert_Dec_Status,
                Orig_Cert_Date, Orig_Cert_Status, Cancel_Req_Date, Cancel_Req_Status, PPS_Num_Date, PPS_Num_Status,
                Afford_State_Date, Afford_State_Status, Identification_Date, Identification_Status, InsurerId,
                Digital_Journey_Date, Digital_Journey_Status, Finance_Form_Date, Finance_Form_Status,
                Nam_Driv_Lic_Date2, Nam_Driv_Lic_Status2, Nam_Driv_Lic_Date3, Nam_Driv_Lic_Status3,
                Nam_Driv_Lic_Date4, Nam_Driv_Lic_Status4, Nam_Driv_Lic_Date5, Nam_Driv_Lic_Status5,
                Campaign, Day1SendDate, Day7SendDate, Day14SendDate, Day20SendDate, CreateDate
        from    (
                select  *, nullif(upper(trim(PolicyCode)), '') as PolicyCodeClean,
                        row_number() over (partition by FileDate, upper(trim(PolicyCode)) order by CreateDate desc) as rn
                from    ${ods.edw}.exp_mychill_chase_daily_snapshot_motorvan
                where   CreateDate >= date '${ods.cutoff_date}'
                ) m
        where   rn = 1
        ) x

## Quote


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_mfq_query_rows                                                       */
/*  Description : MFQ quote query rows, contact details already hashed, times in UTC, only the columns  */
/*              : ods uses.                                                                             */
/*              :                                                                                       */
/*              : Reads: dlk.mfq_quotequery                                                             */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_mfq_query_rows as
select  q.QuoteQueryId,
        nullif(lower(trim(q.QuoteQueryGuid)), '')                                      as GuidKey,
        q.QuoteQueryGuid,
        q.CreatedDateTime,
        case when q.CreatedDateTime < timestamp '2026-06-12 00:00:00'
             then to_utc_timestamp(q.CreatedDateTime, 'Europe/Dublin')
             else q.CreatedDateTime end                             as CreatedDateTimeUtc,
        nullif(upper(trim(q.ReferenceCode)), '')                                       as ReferenceCode,
        nullif(lower(trim(q.RetrieveSessionToken)), '')                                as RetrieveSessionToken,
        q.RetrievalCount,
        q.StepId,
        case when upper(trim(q.ContactEmail)) = 'GDPR' or upper(trim(q.ContactMobile)) = 'GDPR' then 1 else 0 end
                                                                    as IsGdprWiped,
        ods_hash_email(q.ContactEmail)                                  as EmailKey,
        ods_hash_phone(q.ContactMobile)                                 as MobileKey,
        ods_hash_id(q.VehicleRegistrationNo, '')                            as VehicleRegKey,
        ods_eircode_area(q.AddressPostCode)                             as EircodeArea,
        ods_hash_id(concat_ws(' ', q.AddressLine1, q.AddressLine2), '')     as AddressKey,
        cast(q.CoverStartDate as date)                              as CoverStartDate,
        cast(q.PolicyStartDate as date)                             as PolicyStartDate,
        q.PolicyInceptionDate,
        q.BrokerID,
        q.Gclid,
        q.CoverCode,
        q.CoverVoluntaryExcess,
        q.NCDClaimedYearsEarned,
        q.NCDProtectionRequiredInd,
        q.NCDClaimedBonusProtectionType,
        q.PolicyPreviouslyInsuredInd,
        q.PolicyPreviousInsurer,
        q.ProposerSex,
        q.ProposerMaritalStatus,
        q.HomeownerInd,
        q.VehicleModelCode,
        q.VehicleModelName,
        q.VehicleManufacturedYear,
        q.VehicleFirstRegdYear,
        q.VehicleDateFirstRegistered,
        q.VehicleDatePurchased,
        q.VehicleCubicCapacity,
        q.VehicleBodyType,
        q.VehicleImported,
        q.VehicleSecurityDeviceInd,
        q.VehicleAnnualMilage,
        q.VehiclePleasureMileage,
        q.VehicleBusinessMileage,
        q.VehicleValue,
        q.VehicleAreaRating,
        q.VehicleAreaKeptAt,
        q.VehicleKeptAt,
        q.ServiceCallId,
        q.TestGroupFlag,
        q.VariantFlag,
        q.PricePointType
from    ${ods.dlk}.mfq_quotequery q
where   nullif(trim(q.QuoteQueryGuid), '') is not null

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_mfq_query_latest                                                     */
/*  Description : One row per MFQ guid, its latest query row, with first / last activity and row counts */
/*              : .                                                                                     */
/*              :                                                                                       */
/*              : Reads: stg.odsv2_mfq_query_rows                                                       */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_mfq_query_latest as
select  *
from    (
        select  r.*,
                row_number() over (partition by GuidKey order by CreatedDateTime desc, QuoteQueryId desc) as rn,
                max(QuoteQueryId)       over (partition by GuidKey)     as MaxQuoteQueryId,
                min(CreatedDateTimeUtc) over (partition by GuidKey)     as FirstCreatedDateTimeUtc,
                max(CreatedDateTimeUtc) over (partition by GuidKey)     as LastCreatedDateTimeUtc,
                count(*)                over (partition by GuidKey)     as QueryRowCount,
                max(ReferenceCode)      over (partition by GuidKey)     as AnyReferenceCode
        from    ${ods.stg}.odsv2_mfq_query_rows r
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_mfq_fact_sale                                                        */
/*  Description : Sale and quoted price per MFQ guid from the warehouse MFQ fact, as notebook 02.       */
/*              :                                                                                       */
/*              : Reads: edw.tbl_fact_mfq                                                               */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_mfq_fact_sale as
select  nullif(lower(trim(QuoteQueryGuid)), '')                                        as GuidKey,
        max(SaleMFQ_GWP)                                            as SalePremium,
        max(nullif(upper(trim(PolicyCode)), ''))                                       as PolicyCode,
        max(Rank1CustPriceBOTH)                                     as QuotedPremium
from    ${ods.edw}.tbl_fact_mfq
where   nullif(trim(QuoteQueryGuid), '') is not null
group by nullif(lower(trim(QuoteQueryGuid)), '')

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_quote_ccm                                                            */
/*  Description : QuotationStorage (contact-centre motor) quotes, best returned premium, latest calcula */
/*              : tion.                                                                                 */
/*              :                                                                                       */
/*              : Reads: dlk.ext_quotationstorage_policydetails, dlk.ext_quotationstorage_riskdetails,  */
/*              : dlk.ext_quotationstorage_schemeresult                                                 */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_quote_ccm as
select  *
from    (
        select  cast(a.Quotation_ID as string)                      as Ref,
                a.Date_Calculated                                   as QuoteDateTime,
                cast(c.Policy_Start_Date as date)                   as CoverStartDate,
                s.BestPremium                                       as QuotedPremium,
                a.Accepted_Premium                                  as AcceptedPremium,
                nullif(upper(trim(a.Policy_Code)), '')                                 as SourcePolicyCode,
                row_number() over (partition by a.Quotation_ID order by a.Date_Calculated desc, a.`$ExtractDateTime` desc) as rn
        from    ${ods.dlk}.ext_quotationstorage_policydetails   a
        left join ${ods.dlk}.ext_quotationstorage_riskdetails   c   on c.Quotation_ID = a.Quotation_ID
        left join (
                select  Quotation_ID, min(Quote) as BestPremium
                from    ${ods.dlk}.ext_quotationstorage_schemeresult
                where   Decision = 'Quote'
                group by Quotation_ID
                ) s     on s.Quotation_ID = a.Quotation_ID
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_quote_home_web                                                       */
/*  Description : HFQ (home web) quotes, latest version per quote reference, with the sold policy and p */
/*              : rice range.                                                                           */
/*              :                                                                                       */
/*              : Reads: dlk.hfq_quotedetails, dlk.hfq_sales, dlk.hfq_response_quotes                   */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_quote_home_web as
select  *
from    (
        select  nullif(upper(trim(a.QuoteCodeReference)), '')                          as Ref,
                a.CreatedOn                                         as QuoteDateTime,
                cast(a.policyDetails_startDate as date)             as CoverStartDate,
                c.MinPremium                                        as QuotedPremium,
                cast(null as decimal(18,2))                         as AcceptedPremium,
                b.PolicyCode                                        as SourcePolicyCode,
                row_number() over (partition by upper(trim(a.QuoteCodeReference))
                                   order by case when b.PolicyCode is not null then 0 else 1 end,
                                            a.UpdatedOn desc, a.InsertDateTime desc) as rn
        from    ${ods.dlk}.hfq_quotedetails a
        left join (
                select  upper(trim(QuoteCodeReference)) as Ref, max(nullif(upper(trim(PolicyCode)), '')) as PolicyCode
                from    ${ods.dlk}.hfq_sales
                where   Status = 1
                group by upper(trim(QuoteCodeReference))
                ) b     on b.Ref = upper(trim(a.QuoteCodeReference))
        left join (
                select  upper(trim(QuoteCodeReference)) as Ref, min(Quotes_TotalPayable) as MinPremium
                from    ${ods.dlk}.hfq_response_quotes
                where   Quotes_Outcome = 'PremiumReturned'
                group by upper(trim(QuoteCodeReference))
                ) c     on c.Ref = upper(trim(a.QuoteCodeReference))
        where   nullif(trim(a.QuoteCodeReference), '') is not null
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_quote_home_cc                                                        */
/*  Description : Home QS (home, contact centre) quotes, latest transaction per quote (policy) id, chea */
/*              : pest premium.                                                                         */
/*              :                                                                                       */
/*              : Reads: dlk.ext_home_qs_policydetails, dlk.ext_home_qs_quotedetails                    */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_quote_home_cc as
select  *
from    (
        select  upper(a.PolicyId)                                   as Ref,
                a.TransactionDate                                   as QuoteDateTime,
                cast(a.EffectiveStartDate as date)                  as CoverStartDate,
                d.MinPremium                                        as QuotedPremium,
                cast(null as decimal(18,2))                         as AcceptedPremium,
                nullif(upper(trim(a.PolicyCode)), '')                                  as SourcePolicyCode,
                row_number() over (partition by upper(a.PolicyId) order by a.TransactionDate desc, a.`$ExtractDateTime` desc) as rn
        from    ${ods.dlk}.ext_home_qs_policydetails a
        left join (
                select  upper(p.PolicyId) as Ref, min(q.TotalPayable) as MinPremium
                from    ${ods.dlk}.ext_home_qs_policydetails p
                join    ${ods.dlk}.ext_home_qs_quotedetails  q   on q.HomeRiskId = upper(p.RiskId)
                where   q.Outcome = 'Premium Returned' and q.TotalPayable > 0
                group by upper(p.PolicyId)
                ) d     on d.Ref = upper(a.PolicyId)
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_quote_travel                                                         */
/*  Description : Travel quotes, latest update per quote. quote_number 0 is an incomplete quote: its re */
/*              : ference is the hashed email
plus the travel start date.
                              */
/*              :                                                                                       */
/*              : Reads: dlk.ext_travel_quotes                                                          */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_quote_travel as
select  *
from    (
        select  case when q.quote_number <> 0 then cast(q.quote_number as string)
                     else concat('E:', ods_hash_email(q.email), ':', date_format(q.travel_startdate, 'yyyyMMdd'))
                end                                                 as Ref,
                q.travel_quoteDate                                  as QuoteDateTime,
                cast(q.travel_startdate as date)                    as CoverStartDate,
                q.travel_premium                                    as QuotedPremium,
                nullif(upper(trim(q.certificate_reference)), '')                       as CertificateReference,
                row_number() over (partition by case when q.quote_number <> 0 then cast(q.quote_number as string)
                                                     else concat(lower(trim(q.email)), date_format(q.travel_startdate, 'yyyyMMdd')) end
                                   order by q.LastUpdated desc, q.EXT_Travel_QuotesKey desc) as rn
        from    ${ods.dlk}.ext_travel_quotes q
        ) x
where   rn = 1
and     Ref is not null

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_quote_van                                                            */
/*  Description : Van quotes, latest version per quote reference.                                       */
/*              :                                                                                       */
/*              : Reads: dlk.vanquotedetails                                                            */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_quote_van as
select  *
from    (
        select  nullif(upper(trim(v.QuoteReference)), '')                              as Ref,
                to_timestamp(v.QuoteStartDateTime)                  as QuoteDateTime,
                to_date(v.CoverStartDate)                           as CoverStartDate,
                row_number() over (partition by upper(trim(v.QuoteReference)) order by v.CreateDateTime desc, v.PartitionKey desc) as rn
        from    ${ods.dlk}.vanquotedetails v
        where   nullif(trim(v.QuoteReference), '') is not null
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_quote_all                                                            */
/*  Description : All quote systems in one shape, from the cut-off date.                                */
/*              :                                                                                       */
/*              : Reads: stg.odsv2_mfq_query_latest, stg.odsv2_mfq_fact_sale, stg.odsv2_quote_ccm, stg. */
/*              : odsv2_quote_home_web, stg.odsv2_quote_home_cc, stg.odsv2_quote_travel, stg.odsv2_quot */
/*              : e_van                                                                                 */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_quote_all as
select  2 as SourceSystemId, 'MFQ' as SourceSystem, 'Motor' as ProductType, m.GuidKey as Ref,
        m.FirstCreatedDateTimeUtc as QuoteDateTime, m.CoverStartDate, f.QuotedPremium, f.SalePremium as AcceptedPremium,
        f.PolicyCode as SourcePolicyCode, cast(null as string) as CertificateReference
from    ${ods.stg}.odsv2_mfq_query_latest m
left join ${ods.stg}.odsv2_mfq_fact_sale f on f.GuidKey = m.GuidKey
where   m.FirstCreatedDateTimeUtc >= date '${ods.cutoff_date}'
union all
select  9, 'QuotationStorage', 'Motor', Ref, QuoteDateTime, CoverStartDate, QuotedPremium, AcceptedPremium, SourcePolicyCode, null
from    ${ods.stg}.odsv2_quote_ccm where QuoteDateTime >= date '${ods.cutoff_date}'
union all
select  3, 'HFQ', 'Home', Ref, QuoteDateTime, CoverStartDate, QuotedPremium, AcceptedPremium, SourcePolicyCode, null
from    ${ods.stg}.odsv2_quote_home_web where QuoteDateTime >= date '${ods.cutoff_date}'
union all
select  4, 'HomeQS', 'Home', Ref, QuoteDateTime, CoverStartDate, QuotedPremium, AcceptedPremium, SourcePolicyCode, null
from    ${ods.stg}.odsv2_quote_home_cc where QuoteDateTime >= date '${ods.cutoff_date}'
union all
select  8, 'TravelQuotes', 'Travel', Ref, QuoteDateTime, CoverStartDate, QuotedPremium, null, null, CertificateReference
from    ${ods.stg}.odsv2_quote_travel where QuoteDateTime >= date '${ods.cutoff_date}'
union all
select  7, 'VanQuotes', 'Van', Ref, QuoteDateTime, CoverStartDate, null, null, null, null
from    ${ods.stg}.odsv2_quote_van where QuoteDateTime >= date '${ods.cutoff_date}'

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.Quote                                                                        */
/*  Description : Every quote across the quote systems: when it was created, cover start, the quoted an */
/*              : d accepted premium, and the                                                           */
/*              : policy it became. SourcePolicyCode is the code as the quote system holds it; PolicyId */
/*              :  is set only where that code                                                          */
/*              : is a real policy (in ods Policy) - the quote systems also carry non-policy references */
/*              : .                                                                                     */
/*              :                                                                                       */
/*              : Grain: one row per source system and source quote reference                           */
/*              : Business key: SourceSystemId + SourceSystemQuoteReference   Surrogate key: QuoteId    */
/*              : History: current   Cut-off: QuoteDateTime                                             */
/*              : Source: dlk.mfq_quotequery + edw.tbl_fact_mfq, dlk.ext_quotationstorage_*, dlk.hfq_*, */
/*              :  dlk.ext_home_qs_*, dlk.ext_travel_quotes, dlk.vanquotedetails                        */
/*              : Reads: stg.odsv2_quote_all, ods_v2.Policy                                             */
/******************************************************************************************************/
create or replace table ${ods.target}.Quote as
select  row_number() over (order by SourceSystemId, SourceSystemQuoteReference)as QuoteId,
        x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  a.SourceSystemId,
                a.SourceSystem,
                a.ProductType,
                a.Ref                                                           as SourceSystemQuoteReference,
                a.QuoteDateTime,
                a.CoverStartDate,
                a.QuotedPremium,
                a.AcceptedPremium,
                coalesce(a.SourcePolicyCode, t.PolicyCode)                      as SourcePolicyCode,
                coalesce(p.PolicyId, t.PolicyId)                                as PolicyId,
                case when coalesce(p.PolicyId, t.PolicyId) is not null then 1 else 0 end
                                                                                as ConvertedToPolicy
        from    ${ods.stg}.odsv2_quote_all  a
        left join ${ods.target}.Policy      p   on p.PolicyCode = a.SourcePolicyCode
        left join (
                select  InsPolNumber, min(PolicyId) as PolicyId, min(PolicyCode) as PolicyCode
                from    ${ods.target}.Policy
                where   PolicyTypeGroup = 'Travel' and InsPolNumber is not null
                group by InsPolNumber
                ) t     on t.InsPolNumber = a.CertificateReference
        ) x

## QuoteMotorPrice


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteMotorPrice                                                              */
/*  Description : Each insurer's cheapest priced row (FinalYearlyPremium > 0) per MFQ quote, with its f */
/*              : ee and scheme.                                                                        */
/*              :                                                                                       */
/*              : Grain: one row per MFQ quote guid and insurer                                         */
/*              : Business key: GuidKey + Insurer                                                       */
/*              : History: current   Cut-off: none                                                      */
/*              : Source: dlk.mfq_quoteresponses + dlk.mfq_quotesummary                                 */
/*              : Reads: dlk.mfq_quoteresponses, dlk.mfq_quotesummary, ods_v2.Quote                     */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteMotorPrice as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId,
                x.GuidKey,
                x.Insurer,
                x.SchemeName,
                x.FinalYearlyPremium,
                x.Fee,
                x.Premium,
                x.QuoteSummaryId
        from    (
                select  nullif(lower(trim(r.QuoteQueryGuid)), '')                                  as GuidKey,
                        s.Insurer,
                        s.SchemeName,
                        s.FinalYearlyPremium,
                        s.Fee,
                        s.Premium,
                        s.QuoteSummaryId,
                        row_number() over (partition by nullif(lower(trim(r.QuoteQueryGuid)), ''), s.Insurer
                                           order by s.FinalYearlyPremium, s.Fee, s.QuoteSummaryId) as rn
                from    ${ods.dlk}.mfq_quoteresponses   r
                join    ${ods.dlk}.mfq_quotesummary     s   on s.QuoteResponseId = r.QuoteResponseId
                where   s.FinalYearlyPremium > 0
                and     nullif(trim(r.QuoteQueryGuid), '') is not null
                ) x
        join    ${ods.target}.Quote q   on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = x.GuidKey
        where   x.rn = 1
        ) x

## QuoteMotorPayment


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteMotorPayment                                                            */
/*  Description : MFQ online payment attempts tied to the quote (payment -> quote summary -> quote resp */
/*              : onse -> guid; the join RT1                                                            */
/*              : validated on 2026-10-15). PaymentStatus 0 or 5 = successful. Bank and card details ar */
/*              : e not taken.                                                                          */
/*              :                                                                                       */
/*              : Grain: one row per MFQ payment attempt                                                */
/*              : Business key: PaymentId                                                               */
/*              : History: event   Cut-off: none                                                        */
/*              : Source: dlk.mfq_payments + dlk.mfq_quotesummary + dlk.mfq_quoteresponses              */
/*              : Reads: dlk.mfq_payments, dlk.mfq_quotesummary, dlk.mfq_quoteresponses, ods_v2.Quote   */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteMotorPayment as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId,
                p.ID                                                            as PaymentId,
                nullif(lower(trim(r.QuoteQueryGuid)), '')                                          as GuidKey,
                p.QuoteSummaryId,
                p.QuoteSummaryGuid,
                p.PaymentStatus,
                p.Status,
                case when p.PaymentStatus in (0, 5) then 1 else 0 end           as IsSuccessful,
                p.DateAdded                                                     as PaymentDateTimeUtc,
                p.DateUpdated,
                p.IsFullPayment,
                p.IsDirectDebit,
                p.Amount,
                p.DespositAmount                                                as DepositAmount,
                p.MonthlyAmount,
                p.InsurerID,
                p.PolicyStartDate,
                p.IsKeyCareSelected,
                s.Insurer
        from    ${ods.dlk}.mfq_payments         p
        join    ${ods.dlk}.mfq_quotesummary     s   on s.QuoteSummaryId  = p.QuoteSummaryId
        join    ${ods.dlk}.mfq_quoteresponses   r   on r.QuoteResponseId = s.QuoteResponseId
        join    ${ods.target}.Quote             q   on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = nullif(lower(trim(r.QuoteQueryGuid)), '')
        ) x

## QuoteMotorStep


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteMotorStep                                                               */
/*  Description : Every MFQ journey step a quote passed through, in UTC (dlk.mfq_clientsteps CreatedDat */
/*              : eTime is UTC).                                                                        */
/*              :                                                                                       */
/*              : Grain: one row per MFQ client step event                                              */
/*              : Business key: ClientStepId                                                            */
/*              : History: event   Cut-off: StepDateTimeUtc                                             */
/*              : Source: dlk.mfq_clientsteps                                                           */
/*              : Reads: dlk.mfq_clientsteps, ods_v2.Quote                                              */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteMotorStep as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  s.ClientStepId,
                q.QuoteId,
                nullif(lower(trim(s.QuoteQueryGuid)), '')                                          as GuidKey,
                s.StepId,
                s.CoverCode,
                s.CreatedDateTime                                               as StepDateTimeUtc
        from    ${ods.dlk}.mfq_clientsteps s
        left join ${ods.target}.Quote q on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = nullif(lower(trim(s.QuoteQueryGuid)), '')
        where   s.CreatedDateTime >= date '${ods.cutoff_date}'
        ) x

## QuoteMotorBotSession


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteMotorBotSession                                                         */
/*  Description : Imperva bot classification of MFQ sessions (IsKnownBot, ABPAction), per quote.        */
/*              :                                                                                       */
/*              : Grain: one row per MFQ monitoring session record                                      */
/*              : Business key: MonSessionId                                                            */
/*              : History: event   Cut-off: SessionDateTime                                             */
/*              : Source: dlk.mfq_mon_session                                                           */
/*              : Reads: dlk.mfq_mon_session, ods_v2.Quote                                              */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteMotorBotSession as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  m.Id                                                            as MonSessionId,
                q.QuoteId,
                nullif(lower(trim(m.QuoteId)), '')                                                 as GuidKey,
                m.ImpervaSessionId,
                m.`Timestamp`                                                   as SessionDateTime,
                m.IsKnownBot,
                m.ABPAction
        from    ${ods.dlk}.mfq_mon_session m
        left join ${ods.target}.Quote q on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = nullif(lower(trim(m.QuoteId)), '')
        where   m.`Timestamp` >= date '${ods.cutoff_date}'
        ) x

## QuoteMotor


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_vehicle_make                                                         */
/*  Description : Model code -> make, the most frequent make per code in QuotationStorage (ties alphabe */
/*              : tical).                                                                               */
/*              :                                                                                       */
/*              : Reads: dlk.ext_quotationstorage_riskdetails                                           */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_vehicle_make as
select  code, make
from    (
        select  upper(trim(Car_Model_Code))                         as code,
                upper(trim(Car_Make))                               as make,
                row_number() over (partition by upper(trim(Car_Model_Code))
                                   order by count(*) desc, upper(trim(Car_Make))) as r
        from    ${ods.dlk}.ext_quotationstorage_riskdetails
        where   Car_Model_Code is not null and Car_Make is not null
        group by upper(trim(Car_Model_Code)), upper(trim(Car_Make))
        ) m
where   r = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_mfq_proposer                                                         */
/*  Description : The proposer (PRN 1, highest QuoteDriverId) per quote query, age at the quote's cover */
/*              :  start.                                                                               */
/*              :                                                                                       */
/*              : Reads: dlk.mfq_quotedrivers                                                           */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_mfq_proposer as
select  *
from    (
        select  d.QuoteQueryId, d.DateOfBirth, d.DrivesVehicleUse, d.PenaltyPoints, d.ConvictionsInd,
                d.PenaltyPointsIndicator, d.LicenceType, d.IrelandLicenceDate, d.OccupationCode,
                count(*) over (partition by d.QuoteQueryId)                             as ProposerRows,
                row_number() over (partition by d.QuoteQueryId order by d.QuoteDriverId desc) as r
        from    ${ods.dlk}.mfq_quotedrivers d
        where   d.PRN = 1
        ) p
where   r = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_mfq_driver_summary                                                   */
/*  Description : Driver counts and the youngest date of birth per quote query.                         */
/*              :                                                                                       */
/*              : Reads: dlk.mfq_quotedrivers                                                           */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_mfq_driver_summary as
select  QuoteQueryId,
        count(*)                                                    as DriverCount,
        sum(case when PRN <> 1 then 1 else 0 end)                   as NamedDriverCount,
        max(DateOfBirth)                                            as YoungestDateOfBirth
from    ${ods.dlk}.mfq_quotedrivers
group by QuoteQueryId

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_mfq_step_summary                                                     */
/*  Description : Highest step reached and first / last step time per guid.                             */
/*              :                                                                                       */
/*              : Reads: ods_v2.QuoteMotorStep                                                          */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_mfq_step_summary as
select  GuidKey,
        max(StepId)                                                 as MaxStepId,
        min(case when StepId = 1 then StepDateTimeUtc end)          as FirstStep1DateTimeUtc,
        max(StepDateTimeUtc)                                        as LastStepDateTimeUtc
from    ${ods.target}.QuoteMotorStep
group by GuidKey

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_mfq_price_summary                                                    */
/*  Description : Insurers priced and the cheapest price per guid.                                      */
/*              :                                                                                       */
/*              : Reads: ods_v2.QuoteMotorPrice                                                         */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_mfq_price_summary as
select  GuidKey,
        count(*)                                                    as InsurerCount,
        min(FinalYearlyPremium)                                     as CheapestPremium,
        min_by(Fee, FinalYearlyPremium)                             as CheapestFee,
        min_by(Insurer, FinalYearlyPremium)                         as CheapestInsurer
from    ${ods.target}.QuoteMotorPrice
group by GuidKey

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_mfq_payment_summary                                                  */
/*  Description : Online payment attempted / succeeded per guid.                                        */
/*              :                                                                                       */
/*              : Reads: ods_v2.QuoteMotorPayment                                                       */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_mfq_payment_summary as
select  GuidKey,
        1                                                           as PaymentAttempted,
        max(IsSuccessful)                                           as PaidOnline,
        min(case when IsSuccessful = 1 then PaymentDateTimeUtc end) as FirstSuccessfulPaymentDateTimeUtc
from    ${ods.target}.QuoteMotorPayment
group by GuidKey

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteMotor                                                                   */
/*  Description : One row per MFQ car quote: the latest answers, first / last activity (UTC), reference */
/*              :  and retrieve token, hashed                                                           */
/*              : contact keys (email, mobile, registration, address lines 1+2), vehicle (make from the */
/*              :  QuotationStorage model-code                                                          */
/*              : map), proposer and driver summary, step reached,                                      */
/*              : price and payment summary. As ODS_Build_QuoteJourney QuoteMotor, without the SCV key  */
/*              : (that link is ads).                                                                   */
/*              : Latest query only, by design: MFQ occasionally carries a query / driver record over f */
/*              : rom another quote (known                                                              */
/*              : source bug, not fixed), so earlier versions of a guid are not trusted for attributes  */
/*              : or matching.                                                                          */
/*              : Ages are completed years at cover start; no dates of birth are held. IsGdprWiped = em */
/*              : ail or mobile overwritten                                                             */
/*              : with 'GDPR' (non-consenting, non-buying quotes up to 2026-05-06); the row is kept, co */
/*              : nsumers decide.                                                                       */
/*              :                                                                                       */
/*              : Grain: one row per MFQ quote guid                                                     */
/*              : Business key: GuidKey                                                                 */
/*              : History: current   Cut-off: FirstCreatedDateTimeUtc (through Quote)                   */
/*              : Source: dlk.mfq_quotequery + dlk.mfq_quotedrivers + dlk.mfq_clientsteps + ods QuoteMo */
/*              : torPrice / QuoteMotorPayment                                                          */
/*              : Reads: stg.odsv2_mfq_query_latest, ods_v2.Quote, stg.odsv2_vehicle_make, stg.odsv2_mf */
/*              : q_proposer, stg.odsv2_mfq_driver_summary, stg.odsv2_mfq_step_summary, stg.odsv2_mfq_p */
/*              : rice_summary, stg.odsv2_mfq_payment_summary                                           */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteMotor as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId,
                l.GuidKey,
                l.QuoteQueryGuid,
                l.QuoteQueryId                                                  as LatestQuoteQueryId,
                l.MaxQuoteQueryId,
                l.QueryRowCount,
                l.FirstCreatedDateTimeUtc,
                l.LastCreatedDateTimeUtc,
                l.ReferenceCode,
                l.AnyReferenceCode,
                l.RetrieveSessionToken,
                l.RetrievalCount,
                l.IsGdprWiped,
                l.EmailKey,
                l.MobileKey,
                l.VehicleRegKey,
                l.EircodeArea,
                l.AddressKey,
                l.CoverStartDate,
                l.PolicyStartDate,
                l.BrokerID,
                case when nullif(trim(l.Gclid), '') is not null then 1 else 0 end as HasGclid,
                l.CoverCode,
                l.CoverVoluntaryExcess,
                l.NCDClaimedYearsEarned,
                l.NCDProtectionRequiredInd,
                l.NCDClaimedBonusProtectionType,
                l.PolicyPreviouslyInsuredInd,
                l.PolicyPreviousInsurer,
                l.HomeownerInd,
                l.VehicleModelCode,
                l.VehicleModelName,
                mk.make                                                         as VehicleMake,
                l.VehicleManufacturedYear,
                l.VehicleFirstRegdYear,
                l.VehicleCubicCapacity,
                l.VehicleImported,
                l.VehicleAnnualMilage,
                l.VehicleValue,
                l.ProposerSex,
                l.ProposerMaritalStatus,
                floor(months_between(l.CoverStartDate, pr.DateOfBirth) / 12)    as ProposerAgeAtCoverStart,
                pr.DrivesVehicleUse                                             as ProposerDrivesVehicleUse,
                pr.PenaltyPoints                                                as ProposerPenaltyPoints,
                pr.PenaltyPointsIndicator                                       as ProposerPenaltyPointsIndicator,
                pr.ConvictionsInd                                               as ProposerConvictionsInd,
                pr.LicenceType                                                  as ProposerLicenceType,
                pr.IrelandLicenceDate                                           as ProposerIrelandLicenceDate,
                pr.OccupationCode                                               as ProposerOccupationCode,
                coalesce(pr.ProposerRows, 0)                                    as ProposerRows,
                coalesce(d.DriverCount, 0)                                      as DriverCount,
                coalesce(d.NamedDriverCount, 0)                                 as NamedDriverCount,
                floor(months_between(l.CoverStartDate, d.YoungestDateOfBirth) / 12) as YoungestDriverAgeAtCoverStart,
                st.MaxStepId,
                st.FirstStep1DateTimeUtc,
                st.LastStepDateTimeUtc,
                coalesce(ps.InsurerCount, 0)                                    as InsurerCount,
                ps.CheapestPremium,
                ps.CheapestFee,
                ps.CheapestInsurer,
                coalesce(py.PaymentAttempted, 0)                                as PaymentAttempted,
                coalesce(py.PaidOnline, 0)                                      as PaidOnline,
                py.FirstSuccessfulPaymentDateTimeUtc,
                l.TestGroupFlag,
                l.VariantFlag,
                l.PricePointType
        from    ${ods.stg}.odsv2_mfq_query_latest       l
        join    ${ods.target}.Quote                     q   on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = l.GuidKey
        left join ${ods.stg}.odsv2_vehicle_make         mk  on mk.code = upper(trim(l.VehicleModelCode))
        left join ${ods.stg}.odsv2_mfq_proposer         pr  on pr.QuoteQueryId = l.MaxQuoteQueryId
        left join ${ods.stg}.odsv2_mfq_driver_summary   d   on d.QuoteQueryId = l.MaxQuoteQueryId
        left join ${ods.stg}.odsv2_mfq_step_summary     st  on st.GuidKey = l.GuidKey
        left join ${ods.stg}.odsv2_mfq_price_summary    ps  on ps.GuidKey = l.GuidKey
        left join ${ods.stg}.odsv2_mfq_payment_summary  py  on py.GuidKey = l.GuidKey
        ) x

## QuoteMotorSummary


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteMotorSummary                                                            */
/*  Description : The warehouse's per-quote MFQ attributes the assets use: first / last device, marketi */
/*              : ng channel, medium and                                                                */
/*              : campaign, quote type and counts, retrievals, sale. Latest EffectiveDate per guid (2,5 */
/*              : 83 guids span two                                                                     */
/*              : month-ends). Rank1Prem* = the cheapest insurer premium by cover (Comp / TPFT) and NCD */
/*              :  protection (NCB = full                                                               */
/*              : protection), as the warehouse holds them. To be derived from dlk with the DWH logic l */
/*              : ater; edw is then                                                                     */
/*              : reconciliation only.                                                                  */
/*              :                                                                                       */
/*              : Grain: one row per MFQ quote guid (latest month-end EffectiveDate)                    */
/*              : Business key: GuidKey                                                                 */
/*              : History: current   Cut-off: CreatedDateTime                                           */
/*              : Source: edw.tbl_fact_mfq                                                              */
/*              : Reads: edw.tbl_fact_mfq, ods_v2.Quote                                                 */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteMotorSummary as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId,
                m.GuidKey,
                m.EffectiveDate,
                m.CreatedDateTime,
                m.QuoteType,
                m.QQ_All,
                m.FQ_All,
                m.FQ_Orig,
                m.FQ_Retd,
                m.FirstClickFlag,
                m.LastClickFlag,
                m.FirstDevice,
                m.LastDevice,
                m.FirstMktChannel,
                m.LastMktChannel,
                m.FirstMedium,
                m.LastMedium,
                m.FirstCampaign,
                m.LastCampaign,
                m.CampaignName,
                case when nullif(trim(m.GCLID), '') is not null then 1 else 0 end   as HasGclid,
                m.ViewedCoverType,
                m.ViewedBonusProtectionType,
                m.MaxStepId,
                m.MaxStep,
                m.LastStepId,
                m.RetrievalCount,
                nullif(upper(trim(m.RetrievalRef)), '')                                            as RetrievalRef,
                m.ActiveCustomerQuote,
                m.InsurersQuoted,
                m.Rank1PremComp,
                m.Rank1PremTPFT,
                m.Rank1PremCompNCB,
                m.Rank1PremTPFTNCB,
                m.Rank1PremBOTH,
                m.Rank1InsComp,
                m.Rank1InsTPFT,
                m.RiskProfileBand,
                m.QuoteDriverID1,
                m.QuoteDriverID2,
                nullif(upper(trim(m.PolicyCode)), '')                                              as PolicyCode,
                m.ReportingSaleType,
                m.ReportingSaleCategory,
                m.MFQ_Sale,
                m.MFQ_Sale_Type,
                m.SaleMFQ_GWP,
                m.SaleMFQ_Fee,
                m.ImpervaABPaction,
                m.ImpervaKnownBot,
                m.ScraperFlag,
                ods_hash_email(m.CustomerReference)                                 as CustomerReferenceEmailKey
        from    (
                select  *, nullif(lower(trim(QuoteQueryGuid)), '') as GuidKey,
                        row_number() over (partition by nullif(lower(trim(QuoteQueryGuid)), '') order by EffectiveDate desc) as rn
                from    ${ods.edw}.tbl_fact_mfq
                where   nullif(trim(QuoteQueryGuid), '') is not null
                ) m
        left join ${ods.target}.Quote q on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = m.GuidKey
        where   m.rn = 1
        and     m.CreatedDateTime >= date '${ods.cutoff_date}'
        ) x

## QuoteMotorDriver


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_ccm_driver                                                           */
/*  Description : QuotationStorage drivers, latest extract per quotation and PRN.                       */
/*              :                                                                                       */
/*              : Reads: dlk.ext_quotationstorage_driver                                                */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_ccm_driver as
select  *
from    (
        select  d.*, row_number() over (partition by d.Quotation_ID, d.Driver_PRN order by d.`$ExtractDateTime` desc) as rn
        from    ${ods.dlk}.ext_quotationstorage_driver d
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteMotorDriver                                                             */
/*  Description : Drivers on each motor quote: MFQ (the drivers of the guid's highest QuoteQueryId; whe */
/*              : re MFQ holds two drivers with                                                         */
/*              : the same PRN on a query, the highest QuoteDriverId, as the proposer rule) and Quotati */
/*              : onStorage. Licence                                                                    */
/*              : number is a salted hash (the default 123456 becomes null); names and dates of birth a */
/*              : re not held, the age at                                                               */
/*              : cover start is.                                                                       */
/*              :                                                                                       */
/*              : Grain: one row per quote and driver PRN                                               */
/*              : Business key: SourceSystemId + SourceSystemQuoteReference + DriverPRN                 */
/*              : History: current   Cut-off: none                                                      */
/*              : Source: dlk.mfq_quotedrivers (+ mfq_quotequery), dlk.ext_quotationstorage_driver      */
/*              : Reads: stg.odsv2_mfq_query_latest, dlk.mfq_quotedrivers, ods_v2.Quote, stg.odsv2_ccm_ */
/*              : driver                                                                                */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteMotorDriver as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId,
                2                                                               as SourceSystemId,
                l.GuidKey                                                       as SourceSystemQuoteReference,
                d.QuoteQueryId,
                d.QuoteDriverId,
                d.PRN                                                           as DriverPRN,
                floor(months_between(l.CoverStartDate, d.DateOfBirth) / 12)     as DriverAgeAtCoverStart,
                d.Sex                                                           as DriverSex,
                ods_hash_id(d.DriverLicenseNumber, '123456')                        as DriverLicenceKey,
                d.LicenceType                                                   as DriverLicenceType,
                d.LicenceCountry                                                as DriverLicenceCountry,
                cast(d.LicenceRestrictionInd as string)                         as DriverLicenceRestrictions,
                d.IrelandLicenceDate                                            as DriverDateLicenceIssued,
                d.RelationshipToProposer                                        as DriverRelationshipToProposer,
                cast(d.DrivesVehicleDrivingFrequency as string)                 as DriverDrivingFrequency,
                cast(d.FullTimeUseOfOtherCar as string)                         as DriverFullTimeUseOfAnotherCar,
                cast(d.PermResident as string)                                  as DriverPermanentResident,
                d.IrelandResidencyDate                                          as DriverResidentSince,
                cast(d.PrevRefusedCover as string)                              as DriverPreviouslyRefusedCover,
                cast(d.NonMotoringConviction as string)                         as DriverNonMotoringConvictions,
                cast(d.PrevRestrictiveTerms as string)                          as DriverPreviouslyImposedTerms,
                cast(d.ProsecutionPending as string)                            as DriverProsecutionPending,
                cast(d.ResidentWithProposer as string)                          as DriverResidentWithProposer,
                cast(d.ClaimsIndicator as string)                               as DriverClaims,
                cast(d.ConvictionsInd as string)                                as DriverConvictions,
                cast(d.MedicalConditionsInd as string)                          as DriverMedicalConditions,
                cast(d.PenaltyPoints as string)                                 as DriverPenaltyPoints,
                d.OccupationCode                                                as DriverOccupation,
                d.OccupationEmployersBusiness                                   as DriverFullTimeEmployersBusiness,
                d.OccupationEmploymentType                                      as DriverEmploymentType
        from    ${ods.stg}.odsv2_mfq_query_latest   l
        join    (
                select  *, row_number() over (partition by QuoteQueryId, PRN order by QuoteDriverId desc) as rn
                from    ${ods.dlk}.mfq_quotedrivers
                ) d     on d.QuoteQueryId = l.MaxQuoteQueryId and d.rn = 1
        join    ${ods.target}.Quote                 q   on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = l.GuidKey
        union all
        select  q.QuoteId,
                9,
                cast(c.Quotation_ID as string),
                null,
                null,
                c.Driver_PRN,
                c.Driver_Age_at_Effective_Date,
                c.Driver_Sex,
                ods_hash_id(c.Driver_Licence_Number, '123456'),
                c.Driver_Licence_Type,
                c.Driver_Country_of_Issue,
                cast(c.Driver_Licence_Restrictions as string),
                c.Driver_Date_Licence_Issued,
                c.Driver_Relationship_to_Proposer,
                cast(c.Driver_Driving_Frequency as string),
                cast(c.Driver_Full_Time_Use_of_Another_Car as string),
                cast(c.Driver_Permanent_Resident as string),
                c.Driver_Resident_Since,
                cast(c.Driver_Previously_Refused_Cover as string),
                cast(c.Driver_Non_Motoring_convictions as string),
                cast(c.Driver_Previously_Imposed_Terms as string),
                cast(c.Driver_Prosecution_Pending as string),
                cast(c.Driver_Resident_with_Proposer as string),
                cast(c.Driver_Claims as string),
                cast(c.Driver_Convictions as string),
                cast(c.Driver_Medical_Conditions as string),
                cast(c.Driver_Penalty_Points as string),
                c.Driver_Occupation,
                c.Driver_Full_Time_Employers_Business,
                c.Driver_Employment_Type
        from    ${ods.stg}.odsv2_ccm_driver c
        join    ${ods.target}.Quote         q   on q.SourceSystemId = 9 and q.SourceSystemQuoteReference = cast(c.Quotation_ID as string)
        ) x

## QuoteMotorVehicle


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteMotorVehicle                                                            */
/*  Description : The vehicle on each motor quote (MFQ latest answers; QuotationStorage latest extract) */
/*              : . The registration is a                                                               */
/*              : salted hash.                                                                          */
/*              :                                                                                       */
/*              : Grain: one row per motor quote                                                        */
/*              : Business key: SourceSystemId + SourceSystemQuoteReference                             */
/*              : History: current   Cut-off: none                                                      */
/*              : Source: dlk.mfq_quotequery (latest row), dlk.ext_quotationstorage_riskdetails         */
/*              : Reads: stg.odsv2_mfq_query_latest, ods_v2.Quote, stg.odsv2_vehicle_make, dlk.ext_quot */
/*              : ationstorage_riskdetails                                                              */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteMotorVehicle as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId,
                2                                                               as SourceSystemId,
                l.GuidKey                                                       as SourceSystemQuoteReference,
                l.VehicleModelCode                                              as CarModelCode,
                mk.make                                                         as CarMake,
                l.VehicleModelName                                              as CarModelName,
                l.VehicleRegKey,
                l.VehicleDateFirstRegistered                                    as DateFirstRegistered,
                l.VehicleDatePurchased                                          as DateFirstPurchased,
                l.VehicleValue,
                cast(l.VehicleBodyType as string)                               as BodyType,
                case when l.VehicleSecurityDeviceInd = true then 'Yes' else 'No' end as VehicleSecurity,
                l.VehicleCubicCapacity                                          as CarCC,
                cast(l.VehicleAreaRating as string)                             as AreaOfMainUse,
                cast(l.VehicleAreaKeptAt as string)                             as AreaKeptOvernight,
                cast(l.VehicleKeptAt as string)                                 as LocationKeptOvernight,
                l.VehiclePleasureMileage                                        as PrivateMileage,
                l.VehicleBusinessMileage                                        as BusinessMileage
        from    ${ods.stg}.odsv2_mfq_query_latest   l
        join    ${ods.target}.Quote                 q   on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = l.GuidKey
        left join ${ods.stg}.odsv2_vehicle_make     mk  on mk.code = upper(trim(l.VehicleModelCode))
        union all
        select  q.QuoteId,
                9,
                cast(r.Quotation_ID as string),
                r.Car_Model_Code,
                r.Car_Make,
                r.Car_Model_Name,
                ods_hash_id(r.Registration_Number, ''),
                r.Date_First_Registered,
                r.Date_First_Purchased,
                r.Value,
                cast(r.Body_Type as string),
                cast(r.Vehicle_Security as string),
                r.Car_CC,
                cast(r.Area_of_Main_Use as string),
                cast(r.Area_Kept_Overnight as string),
                cast(r.Location_Kept_Overnight as string),
                r.Private_Mileage,
                r.Business_Mileage
        from    (
                select  *, row_number() over (partition by Quotation_ID order by `$ExtractDateTime` desc) as rn
                from    ${ods.dlk}.ext_quotationstorage_riskdetails
                ) r
        join    ${ods.target}.Quote q   on q.SourceSystemId = 9 and q.SourceSystemQuoteReference = cast(r.Quotation_ID as string)
        where   r.rn = 1
        ) x

## QuoteHome


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_quote_home_cc_detail                                                 */
/*  Description : Home QS quote detail, latest transaction per quote id, with specified items and price */
/*              :  range.                                                                               */
/*              :                                                                                       */
/*              : Reads: dlk.ext_home_qs_policydetails, dlk.ext_home_qs_riskdetails, dlk.ext_home_qs_sp */
/*              : ecifieditemsdetails, dlk.ext_home_qs_quotedetails                                     */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_quote_home_cc_detail as
select  *
from    (
        select  upper(a.PolicyId)                                   as Ref,
                a.TransactionDate                                   as QuotationDate,
                a.BusinessProcess                                   as QuoteType,
                cast(a.EffectiveStartDate as date)                  as CoverStartDate,
                cast(add_months(a.EffectiveStartDate, 12) as date)  as CoverEndDate,
                b.ResidenceDate                                     as HomeResidenceStartDate,
                cast(b.OccupiedDuringDate as string)                as OccupiedDuringDay,
                cast(b.NeigbourhoodWatch as string)                 as NeighbourhoodWatch,
                b.ResidenceType,
                cast(b.IsLandlord as string)                        as IsLandlord,
                b.NumberofTimesLetinaYear                           as NumberOfTimesLetInYear,
                b.RoofConstructionType,
                b.NumberofSmokeAlarms                               as NumberOfSmokeAlarms,
                b.HeatingType,
                cast(b.HasBasement as string)                       as HasBasement,
                b.NumberofBedrooms                                  as NumberOfBedrooms,
                b.NumberofBathrooms                                 as NumberOfBathrooms,
                b.PropertyType,
                b.ClaimedYearsNCD,
                c.ItemSumInsuredRequired,
                c.InsuredItemsCount,
                d.MinPremium,
                d.MaxPremium,
                row_number() over (partition by upper(a.PolicyId) order by a.TransactionDate desc, a.`$ExtractDateTime` desc) as rn
        from    ${ods.dlk}.ext_home_qs_policydetails    a
        left join ${ods.dlk}.ext_home_qs_riskdetails    b   on b.HomeRiskId = upper(a.RiskId)
        left join (
                select  r.HomeCoverId, sum(i.ItemSumInsuredRequired) as ItemSumInsuredRequired, count(*) as InsuredItemsCount
                from    ${ods.dlk}.ext_home_qs_riskdetails              r
                join    ${ods.dlk}.ext_home_qs_specifieditemsdetails    i   on i.HomeCoverId = r.HomeCoverId
                where   i.ItemType <> 'Unspecified'
                group by r.HomeCoverId
                ) c     on c.HomeCoverId = b.HomeCoverId
        left join (
                select  upper(p.PolicyId) as Ref, min(q.TotalPayable) as MinPremium, max(q.TotalPayable) as MaxPremium
                from    ${ods.dlk}.ext_home_qs_policydetails p
                join    ${ods.dlk}.ext_home_qs_quotedetails  q   on q.HomeRiskId = upper(p.RiskId)
                where   q.Outcome = 'Premium Returned' and q.TotalPayable > 0
                group by upper(p.PolicyId)
                ) d     on d.Ref = upper(a.PolicyId)
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_quote_home_web_detail                                                */
/*  Description : HFQ quote detail, latest version per quote reference, journey fields and hashed conta */
/*              : ct keys.                                                                              */
/*              :                                                                                       */
/*              : Reads: dlk.hfq_quotedetails                                                           */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_quote_home_web_detail as
select  *
from    (
        select  nullif(upper(trim(a.QuoteCodeReference)), '')                          as Ref,
                a.CreatedOn                                         as QuotationDate,
                a.UpdatedOn,
                'New Business'                                      as QuoteType,
                cast(a.policyDetails_startDate as date)             as CoverStartDate,
                cast(add_months(a.policyDetails_startDate, 12) as date) as CoverEndDate,
                cast(a.residencyDetails_isOccupiedDuringTheDay as string) as OccupiedDuringDay,
                cast(a.propertyDetails_securityDetails_hasNeighbourhoodWatch as string) as NeighbourhoodWatch,
                case a.residencyDetails_typeOf
                     when 'HolidayHomeFamilyUse' then 'Holiday Home - Family Use'
                     when 'HolidayHomeLetOut'    then 'Holiday Home - Let Out'
                     when 'OwnerOccupied'        then 'Owner Occupied'
                     when 'RentedCouple'         then 'Rented - Couple'
                     when 'RentedFamily'         then 'Rented - Family'
                     when 'RentedFivePeople'     then 'Rented - 5 People'
                     when 'RentedFourPeople'     then 'Rented - 4 People'
                     when 'RentedOnePerson'      then 'Rented - 1 Person'
                     when 'RentedSixPeople'      then 'Rented - 6 People'
                     when 'RentedStudents'       then 'Rented - Students'
                     when 'RentedTenPeople'      then 'Rented - 10 People'
                     when 'RentedThreePeople'    then 'Rented - 3 People'
                     when 'RentedTwoPeople'      then 'Rented - 2 People'
                     when 'Unoccupied'           then 'Unoccupied'
                     else 'Unknown' end                             as ResidenceType,
                cast(a.ResidencyDetails_IsLandlord as string)       as IsLandlord,
                case when a.propertyDetails_roofConstructionDetails_is100PercentStandardMaterial = 1 then 'Standard'
                     when a.PropertyDetails_RoofConstructionDetails_TypeOfNonStandardMaterialUsed is not null
                          then a.PropertyDetails_RoofConstructionDetails_TypeOfNonStandardMaterialUsed
                     else 'NotListed' end                           as RoofConstructionType,
                a.propertyDetails_securityDetails_numberOfSmokeAlarms as NumberOfSmokeAlarms,
                a.propertyDetails_typeOfHeatingSystem               as HeatingType,
                cast(a.propertyDetails_hasABasement as string)      as HasBasement,
                a.propertyDetails_numberOfBedrooms                  as NumberOfBedrooms,
                a.propertyDetails_numberOfBathrooms                 as NumberOfBathrooms,
                a.propertyDetails_typeOf                            as PropertyType,
                a.coverDetails_contentsCoverAmount                  as ItemSumInsuredRequired,
                try_cast(a.RetrieveCount as int)                    as RetrieveCount,
                a.MaxStep,
                a.LastStep,
                a.proposer_previousPolicyDetails_ExpiryDate         as PreviousPolicyExpiryDate,
                a.ImpervaIsBot,
                a.ImpervaSuspectedBot,
                ods_hash_email(a.proposer_email)                        as EmailKey,
                ods_hash_phone(a.proposer_mobilePhoneNumber)            as MobileKey,
                ods_eircode_area(a.propertyDetails_address_eircode)     as EircodeArea,
                ods_hash_id(concat_ws(' ', a.propertyDetails_Address_Line1, a.propertyDetails_Address_Line2), '') as AddressKey,
                row_number() over (partition by upper(trim(a.QuoteCodeReference))
                                   order by a.UpdatedOn desc, a.InsertDateTime desc) as rn
        from    ${ods.dlk}.hfq_quotedetails a
        where   nullif(trim(a.QuoteCodeReference), '') is not null
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteHome                                                                    */
/*  Description : Home quote detail from the web (HFQ, 3) and the contact centre (Home QS, 4), as noteb */
/*              : ook 02, one row per quote                                                             */
/*              : (the web side keeps its latest version). Web-only journey fields (retrievals, steps,  */
/*              : bot flags) and hashed contact                                                         */
/*              : keys are added; the Eircode is reduced to its routing key.                            */
/*              :                                                                                       */
/*              : Grain: one row per home quote                                                         */
/*              : Business key: SourceSystemId + SourceSystemQuoteReference                             */
/*              : History: current   Cut-off: none                                                      */
/*              : Source: dlk.hfq_quotedetails (+ hfq_sales, hfq_response_quotes), dlk.ext_home_qs_* (c */
/*              : ontact centre)                                                                        */
/*              : Reads: stg.odsv2_quote_home_cc_detail, ods_v2.Quote, stg.odsv2_quote_home_web_detail  */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteHome as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId, 4 as SourceSystemId, c.Ref as SourceSystemQuoteReference, q.SourcePolicyCode, q.PolicyId,
                c.QuotationDate, cast(null as timestamp) as UpdatedOn, c.QuoteType, c.CoverStartDate, c.CoverEndDate,
                c.HomeResidenceStartDate, c.OccupiedDuringDay, c.NeighbourhoodWatch, c.ResidenceType, c.IsLandlord,
                c.NumberOfTimesLetInYear, c.RoofConstructionType, c.NumberOfSmokeAlarms, c.HeatingType, c.HasBasement,
                c.NumberOfBedrooms, c.NumberOfBathrooms, cast(c.PropertyType as string) as PropertyType, c.ClaimedYearsNCD,
                c.ItemSumInsuredRequired, c.InsuredItemsCount, c.MinPremium, c.MaxPremium,
                cast(null as int) as RetrieveCount, cast(null as string) as MaxStep, cast(null as string) as LastStep,
                cast(null as date) as PreviousPolicyExpiryDate, cast(null as string) as ImpervaIsBot,
                cast(null as string) as ImpervaSuspectedBot, cast(null as string) as EmailKey, cast(null as string) as MobileKey,
                cast(null as string) as EircodeArea, cast(null as string) as AddressKey
        from    ${ods.stg}.odsv2_quote_home_cc_detail c
        join    ${ods.target}.Quote q on q.SourceSystemId = 4 and q.SourceSystemQuoteReference = c.Ref
        union all
        select  q.QuoteId, 3, w.Ref, q.SourcePolicyCode, q.PolicyId,
                w.QuotationDate, w.UpdatedOn, w.QuoteType, w.CoverStartDate, w.CoverEndDate,
                cast(null as date), w.OccupiedDuringDay, w.NeighbourhoodWatch, w.ResidenceType, w.IsLandlord,
                cast(null as int), w.RoofConstructionType, w.NumberOfSmokeAlarms, w.HeatingType, w.HasBasement,
                w.NumberOfBedrooms, w.NumberOfBathrooms, cast(w.PropertyType as string), cast(null as int),
                w.ItemSumInsuredRequired, cast(null as bigint), q.QuotedPremium, cast(null as decimal(18,2)),
                w.RetrieveCount, cast(w.MaxStep as string), cast(w.LastStep as string),
                cast(w.PreviousPolicyExpiryDate as date), cast(w.ImpervaIsBot as string),
                cast(w.ImpervaSuspectedBot as string), w.EmailKey, w.MobileKey,
                w.EircodeArea, w.AddressKey
        from    ${ods.stg}.odsv2_quote_home_web_detail w
        join    ${ods.target}.Quote q on q.SourceSystemId = 3 and q.SourceSystemQuoteReference = w.Ref
        ) x

## PolicyHomeQsEvent


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyHomeQsEvent                                                            */
/*  Description : Every Home QS (contact-centre home) policy transaction: business process, event type  */
/*              : and dates, per risk. Feeds                                                            */
/*              : the home renewal and home claim events (notebook 04 cells 197-199, 273-275), which jo */
/*              : in it to the claims on the                                                            */
/*              : risk id. HomeQsPolicyId is the Home QS quote / policy id (= Quote.SourceSystemQuoteRe */
/*              : ference for system 4).                                                                */
/*              :                                                                                       */
/*              : Grain: one row per Home QS transaction (risk id)                                      */
/*              : Business key: HomeRiskId                                                              */
/*              : History: event   Cut-off: TransactionDate                                             */
/*              : Source: dlk.ext_home_qs_policydetails                                                 */
/*              : Reads: dlk.ext_home_qs_policydetails, ods_v2.Quote, ods_v2.Policy                     */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyHomeQsEvent as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  upper(p.RiskId)                                                 as HomeRiskId,
                upper(p.PolicyId)                                               as HomeQsPolicyId,
                q.QuoteId,
                pl.PolicyId,
                nullif(upper(trim(p.PolicyCode)), '')                                              as PolicyCode,
                p.BusinessProcess,
                p.EventType,
                p.TransactionDate,
                p.EffectiveStartDate,
                p.EffectiveEndDate,
                p.RenewalDate,
                p.InceptionDate,
                p.WithBrokerFromDate,
                nullif(upper(trim(p.PolicyNumber)), '')                                            as PolicyNumber,
                nullif(upper(trim(p.PolicyReference)), '')                                         as PolicyReference,
                p.InsuranceCompany,
                p.BrokerCompany,
                p.AgencyName,
                p.`$ExtractDateTime`                                            as ExtractDateTime
        from    ${ods.dlk}.ext_home_qs_policydetails    p
        left join ${ods.target}.Quote               q   on q.SourceSystemId = 4 and q.SourceSystemQuoteReference = upper(p.PolicyId)
        left join ${ods.target}.Policy              pl  on pl.PolicyCode = upper(trim(p.PolicyCode))
        where   p.TransactionDate >= date '${ods.cutoff_date}'
        ) x

## QuoteHomePrice


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteHomePrice                                                               */
/*  Description : Each insurer / scheme price returned to an HFQ home quote, with excesses, levy and ou */
/*              : tcome.                                                                                */
/*              :                                                                                       */
/*              : Grain: one row per HFQ quote response row (insurer / scheme price)                    */
/*              : Business key: ResponseQuoteId                                                         */
/*              : History: event   Cut-off: DateCreated                                                 */
/*              : Source: dlk.hfq_response_quotes                                                       */
/*              : Reads: dlk.hfq_response_quotes, ods_v2.Quote                                          */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteHomePrice as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  r.unq_id                                                        as ResponseQuoteId,
                q.QuoteId,
                nullif(upper(trim(r.QuoteCodeReference)), '')                                      as QuoteCodeReference,
                nullif(upper(trim(r.PolicyCode)), '')                                              as PolicyCode,
                r.Quotes_InsurerName                                            as InsurerName,
                r.Quotes_SchemeName                                             as SchemeName,
                r.Quotes_SchemeCode                                             as SchemeCode,
                r.Quotes_SchemeCategory                                         as SchemeCategory,
                r.Quotes_Premium                                                as Premium,
                r.Quotes_TotalPayable                                           as TotalPayable,
                r.Quotes_TotalExcess                                            as TotalExcess,
                r.Quotes_CompulsoryExcess                                       as CompulsoryExcess,
                r.Quotes_LevyAmount                                             as LevyAmount,
                r.Quotes_Outcome                                                as Outcome,
                r.Quotes_IsIndicative                                           as IsIndicative,
                r.Quotes_DateCreated                                            as DateCreated,
                r.Quotes_DateValidTo                                            as DateValidTo,
                r.Quotes_CoverEffectiveFrom                                     as CoverEffectiveFrom,
                r.Quotes_CoverEffectiveTo                                       as CoverEffectiveTo
        from    ${ods.dlk}.hfq_response_quotes r
        left join ${ods.target}.Quote q on q.SourceSystemId = 3 and q.SourceSystemQuoteReference = upper(trim(r.QuoteCodeReference))
        where   r.Quotes_DateCreated >= date '${ods.cutoff_date}'
        ) x

## QuoteHomePaymentRequest


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteHomePaymentRequest                                                      */
/*  Description : HFQ payment requests sent to the payment provider. Payer name and billing address are */
/*              :  not taken; the payer email                                                           */
/*              : is a salted hash.                                                                     */
/*              :                                                                                       */
/*              : Grain: one row per HFQ payment request (global payment reference)                     */
/*              : Business key: GlobalPaymentReference                                                  */
/*              : History: event   Cut-off: InsertDateTime                                              */
/*              : Source: dlk.hfq_paymentrequest_request                                                */
/*              : Reads: dlk.hfq_paymentrequest_request, ods_v2.Quote                                   */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteHomePaymentRequest as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  x.GlobalPaymentReference, q.QuoteId, x.QuoteReference, x.Reference, x.TemporaryQuoteId, x.CacheId, x.Amount,
                x.DepositAmount, x.MonthlyAmount, x.IsFullPayment, x.PolicyStartDate, x.SchemeCode, x.BrokerID, x.RequestType,
                x.InsertDateTime, x.PayerEmailKey
        from    (
                select  globalPayment_reference                                 as GlobalPaymentReference,
                        nullif(upper(trim(QuoteReference)), '')                                    as QuoteReference,
                        reference                                               as Reference,
                        TemporaryQuoteId,
                        CacheId,
                        amount                                                  as Amount,
                        depositAmount                                           as DepositAmount,
                        monthlyAmount                                           as MonthlyAmount,
                        isFullPayment                                           as IsFullPayment,
                        policyStartDate                                         as PolicyStartDate,
                        schemeCode                                              as SchemeCode,
                        BrokerID,
                        type                                                    as RequestType,
                        InsertDateTime,
                        ods_hash_email(payer_email)                                 as PayerEmailKey,
                        row_number() over (partition by globalPayment_reference order by InsertDateTime desc) as rn
                from    ${ods.dlk}.hfq_paymentrequest_request
                where   nullif(trim(globalPayment_reference), '') is not null
                and     InsertDateTime >= date '${ods.cutoff_date}'
                ) x
        left join ${ods.target}.Quote q on q.SourceSystemId = 3 and q.SourceSystemQuoteReference = x.QuoteReference
        where   x.rn = 1
        ) x

## QuoteHomePaymentResponse


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteHomePaymentResponse                                                     */
/*  Description : HFQ payment provider responses: status, result code and amount per transaction. Card  */
/*              : details are not taken.                                                                */
/*              :                                                                                       */
/*              : Grain: one row per HFQ payment response (provider transaction id)                     */
/*              : Business key: PaymentResponseId                                                       */
/*              : History: event   Cut-off: InsertDateTime                                              */
/*              : Source: dlk.hfq_paymentresponse_response                                              */
/*              : Reads: dlk.hfq_paymentresponse_response, ods_v2.Quote                                 */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteHomePaymentResponse as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  x.PaymentResponseId, q.QuoteId, x.QuoteReference, x.Reference, x.TimeCreated, x.ResponseType, x.Status, x.Channel,
                x.Amount, x.Currency, x.PaymentMethodResult, x.PaymentMethodMessage, x.ActionType, x.ActionResultCode,
                x.ActionTimeCreated, x.InsertDateTime
        from    (
                select  id                                                      as PaymentResponseId,
                        nullif(upper(trim(QuoteReference)), '')                                    as QuoteReference,
                        reference                                               as Reference,
                        time_created                                            as TimeCreated,
                        type                                                    as ResponseType,
                        status                                                  as Status,
                        channel                                                 as Channel,
                        amount                                                  as Amount,
                        currency                                                as Currency,
                        payment_method_result                                   as PaymentMethodResult,
                        payment_method_message                                  as PaymentMethodMessage,
                        action_type                                             as ActionType,
                        action_result_code                                      as ActionResultCode,
                        action_time_created                                     as ActionTimeCreated,
                        InsertDateTime,
                        row_number() over (partition by id order by InsertDateTime desc) as rn
                from    ${ods.dlk}.hfq_paymentresponse_response
                where   nullif(trim(id), '') is not null
                and     InsertDateTime >= date '${ods.cutoff_date}'
                ) x
        left join ${ods.target}.Quote q on q.SourceSystemId = 3 and q.SourceSystemQuoteReference = x.QuoteReference
        where   x.rn = 1
        ) x

## QuoteHomeSale


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteHomeSale                                                                */
/*  Description : HFQ web sales feed: the quote reference, the code it was sold under, sale date and st */
/*              : atus (Status 1 = sold, as                                                             */
/*              : notebook 02; other values undocumented). Latest load per sale id: the feed reloads re */
/*              : cords, and the earlier copies                                                         */
/*              : carry no status. PolicyCode is as the feed holds it and is empty for some records (a  */
/*              : later reload may fill it);                                                            */
/*              : linking a sale to a policy is derived in ads (warehouse HEQ matching: left(PolicyCode */
/*              : , 6) as client code, then                                                             */
/*              : email / mobile / address), not here. Payer and payment JSON are not taken.            */
/*              :                                                                                       */
/*              : Grain: one row per HFQ sale record id (latest load)                                   */
/*              : Business key: SaleRecordId                                                            */
/*              : History: event (no cut-off; small)   Cut-off: none                                    */
/*              : Source: dlk.hfq_sales                                                                 */
/*              : Reads: dlk.hfq_sales, ods_v2.Quote, ods_v2.Policy                                     */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteHomeSale as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  x.SaleRecordId, q.QuoteId, p.PolicyId, x.QuoteCodeReference, x.PolicyCode, x.SourceQuoteId, x.TemporaryQuoteId,
                x.CacheID, x.GlobalPaymentReference, x.SchemeCode, x.Status, x.SaleDate, x.InsertDateTime
        from    (
                select  id                                                      as SaleRecordId,
                        nullif(upper(trim(QuoteCodeReference)), '')                                as QuoteCodeReference,
                        nullif(upper(trim(PolicyCode)), '')                                        as PolicyCode,
                        quoteid                                                 as SourceQuoteId,
                        TemporaryQuoteId,
                        CacheID,
                        GlobalPaymentReference,
                        SchemeCode,
                        Status,
                        SaleDate,
                        InsertDateTime,
                        row_number() over (partition by id order by InsertDateTime desc, Status desc) as rn
                from    ${ods.dlk}.hfq_sales
                where   nullif(trim(id), '') is not null
                ) x
        left join ${ods.target}.Quote   q   on q.SourceSystemId = 3 and q.SourceSystemQuoteReference = x.QuoteCodeReference
        left join ${ods.target}.Policy  p   on p.PolicyCode = x.PolicyCode
        where   x.rn = 1
        ) x

## QuoteTravel


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteTravel                                                                  */
/*  Description : Travel quote detail, latest update per quote. Names, address, date of birth are not t */
/*              : aken; email and mobile are                                                            */
/*              : salted hashes. MarketingConsent_* are consent flags, not contact details.             */
/*              :                                                                                       */
/*              : Grain: one row per travel quote (quote number; incomplete quotes by hashed email + tr */
/*              : avel date)                                                                            */
/*              : Business key: SourceSystemQuoteReference                                              */
/*              : History: current   Cut-off: QuoteDate (through Quote)                                 */
/*              : Source: dlk.ext_travel_quotes                                                         */
/*              : Reads: dlk.ext_travel_quotes, ods_v2.Quote                                            */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteTravel as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId,
                q.SourceSystemQuoteReference,
                q.PolicyId,
                t.quote_number                                                  as QuoteNumber,
                t.travel_certificatestatus                                      as CertificateStatus,
                nullif(upper(trim(t.certificate_reference)), '')                                   as CertificateReference,
                t.cover_level                                                   as CoverLevel,
                t.destination                                                   as Destination,
                t.no_adults                                                     as NoAdults,
                t.no_children                                                   as NoChildren,
                t.no_infants                                                    as NoInfants,
                t.travel_premium                                                as TravelPremium,
                t.travel_quoteDate                                              as QuoteDate,
                t.travel_startdate                                              as TravelStartDate,
                t.travel_enddate                                                as TravelEndDate,
                t.travel_purchaseDate                                           as PurchaseDate,
                t.business_type                                                 as BusinessType,
                t.MarketingConsent_Email,
                t.MarketingConsent_SMS,
                t.Marketing_Opted_In                                            as MarketingOptedIn,
                t.reseller                                                      as Reseller,
                t.campaign                                                      as Campaign,
                t.phi                                                           as Phi,
                t.ski                                                           as Ski,
                t.has_medical                                                   as HasMedical,
                t.travel_health_insurance                                       as TravelHealthInsurance,
                t.duration                                                      as Duration,
                t.customer_age                                                  as CustomerAge,
                t.LastUpdated,
                ods_hash_email(t.email)                                             as EmailKey,
                ods_hash_phone(t.mobile_number)                                     as MobileKey,
                ods_hash_id(t.user_id, '')                                              as UserIdKey
        from    (
                select  *,
                        case when quote_number <> 0 then cast(quote_number as string)
                             else concat('E:', ods_hash_email(email), ':', date_format(travel_startdate, 'yyyyMMdd')) end as Ref,
                        row_number() over (partition by case when quote_number <> 0 then cast(quote_number as string)
                                                             else concat(lower(trim(email)), date_format(travel_startdate, 'yyyyMMdd')) end
                                           order by LastUpdated desc, EXT_Travel_QuotesKey desc) as rn
                from    ${ods.dlk}.ext_travel_quotes
                ) t
        join    ${ods.target}.Quote q   on q.SourceSystemId = 8 and q.SourceSystemQuoteReference = t.Ref
        where   t.rn = 1
        ) x

## QuoteVan


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteVan                                                                     */
/*  Description : Van quote answers (latest PartitionKey per RowKey, as notebook 02). Names, date of bi */
/*              : rth, address and the full                                                             */
/*              : Eircode are not taken; email and phone are salted hashes, the Eircode its routing key */
/*              : . IsTestEmail flags test /                                                            */
/*              : internal email domains (att.net, @app.net, @chill.ie, @test), set before hashing.     */
/*              :                                                                                       */
/*              : Grain: one row per van quote record (RowKey), its latest partition                    */
/*              : Business key: RowKey                                                                  */
/*              : History: current   Cut-off: none                                                      */
/*              : Source: dlk.vanquotedetails                                                           */
/*              : Reads: dlk.vanquotedetails, ods_v2.Quote                                              */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteVan as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId,
                v.RowKey,
                v.PartitionKey,
                nullif(upper(trim(v.QuoteReference)), '')                                          as QuoteReference,
                to_timestamp(v.QuoteStartDateTime)                              as QuoteStartDateTime,
                v.QuoteType,
                to_date(v.CoverStartDate)                                       as CoverStartDate,
                v.CoverType,
                v.AnnualMileage,
                v.AreaofUse                                                     as AreaOfUse,
                v.SubAreaofUse                                                  as SubAreaOfUse,
                v.CrewTwinCab,
                v.CurrentVanNCD,
                v.CurrentVanNCDNumYears,
                v.PrivateCarNCD,
                v.PrivateCarNCDNumYears,
                v.SecondCarNCD,
                v.SecondCarNCDNumYears,
                v.NCB,
                v.LicenceType,
                v.LicenceYearsHeld,
                v.NameOnCommercialVehicle,
                v.NameOnCommercialVehicleNumYears,
                v.NameOnPrivateCar,
                v.NameOnPrivateCarNumYears,
                v.NumOfAdditionalDrivers,
                v.Occupation,
                v.PenaltyPoints,
                v.VehicleValue,
                v.VoluntaryExcessAmount,
                v.WindscreenCover,
                v.YearRegistered,
                v.RiskAddressCounty                                             as RiskCounty,
                floor(months_between(to_date(v.CoverStartDate), to_date(v.DOB)) / 12) as ProposerAgeAtCoverStart,
                v.SessionID,
                v.CreateDateTime,
                ods_hash_email(v.Email)                                             as EmailKey,
                case when lower(v.Email) like '%att.net' or lower(v.Email) like '%@app.net%'
                       or lower(v.Email) like '%@chill.ie' or lower(v.Email) like '%@test%'
                     then 1 else 0 end                                          as IsTestEmail,    -- notebook 04 cell 343
                ods_hash_phone(v.PhoneNumber)                                       as PhoneKey,
                ods_eircode_area(v.Eircode)                                         as EircodeArea,
                ods_hash_id(concat_ws(' ', v.RiskAddressLine1, v.RiskAddressLine2), '') as AddressKey
        from    (
                select  *, row_number() over (partition by RowKey order by PartitionKey desc) as rn
                from    ${ods.dlk}.vanquotedetails
                ) v
        left join ${ods.target}.Quote q on q.SourceSystemId = 7 and q.SourceSystemQuoteReference = upper(trim(v.QuoteReference))
        where   v.rn = 1
        and     to_timestamp(v.QuoteStartDateTime) >= date '${ods.cutoff_date}'
        ) x

## QuoteVanJourney


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteVanJourney                                                              */
/*  Description : Van quote journey page events: insurer shown, display status / errors, retrieval, GCL */
/*              : ID. The email address is a                                                            */
/*              : salted hash. Latest load per event.                                                   */
/*              :                                                                                       */
/*              : Grain: one row per van quote page event (quote, session, timestamp, click code)       */
/*              : Business key: QuoteReference + SessionId + EventTimestamp + ClickCode                 */
/*              : History: event   Cut-off: CreateDateTime                                              */
/*              : Source: dlk.vanquotejourney                                                           */
/*              : Reads: dlk.vanquotejourney, ods_v2.Quote                                              */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteVanJourney as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId, x.QuoteReference, x.SessionId, x.EventTimestamp, x.ClickCode, x.QuoteStartDateTime, x.Insurer,
                x.PageDisplayStatus, x.PageDisplayErrorCode, x.PageDisplayErrorMessage, x.IsRetrieved, x.HasGclid,
                x.InvalidEmailAddress, x.InvalidDob, x.InvalidQuoteReference, x.CreateDateTime, x.EmailKey
        from    (
                select  nullif(upper(trim(QuoteReference)), '')                                    as QuoteReference,
                        SessionId,
                        `Timestamp`                                             as EventTimestamp,
                        ClickCode,
                        QuoteStartDateTime,
                        Insurer,
                        PageDisplayStatus,
                        PageDisplayErrorCode,
                        ods_redact(PageDisplayErrorMessage)                         as PageDisplayErrorMessage,
                        IsRetrieved,
                        case when nullif(trim(GCLID), '') is not null then 1 else 0 end as HasGclid,
                        InvalidEmailAddress,
                        InvalidDob,
                        InvalidQuoteReference,
                        CreateDateTime,
                        ods_hash_email(EmailAddress)                                as EmailKey,
                        row_number() over (partition by upper(trim(QuoteReference)), SessionId, `Timestamp`, ClickCode
                                           order by CreateDateTime desc) as rn
                from    ${ods.dlk}.vanquotejourney
                where   CreateDateTime >= date '${ods.cutoff_date}'
                ) x
        left join ${ods.target}.Quote q on q.SourceSystemId = 7 and q.SourceSystemQuoteReference = x.QuoteReference
        where   x.rn = 1
        ) x

## QuoteVanPayment


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteVanPayment                                                              */
/*  Description : Van online payment attempts: type, amount, status and errors, scheme and insurer. Lat */
/*              : est load per event. About                                                             */
/*              : half the source rows have no payment timestamp; PaymentDateTime then falls back to th */
/*              : e load time                                                                           */
/*              : (PaymentDateTimeIsLoadTime = 1).                                                      */
/*              :                                                                                       */
/*              : Grain: one row per van payment event (quote, session, time, click code)               */
/*              : Business key: QuoteReference + SessionId + PaymentDateTime + ClickCode                */
/*              : History: event   Cut-off: CreateDateTime                                              */
/*              : Source: dlk.vanpaymentjourney                                                         */
/*              : Reads: dlk.vanpaymentjourney, ods_v2.Quote                                            */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteVanPayment as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId, x.QuoteReference, x.SessionId, x.PaymentDateTime, x.PaymentDateTimeIsLoadTime, x.ClickCode,
                x.QuoteStartDateTime, x.PaymentType,
                x.PaymentAmount, x.PaymentStatus, x.PaymentErrorCode, x.PaymentErrorMessage, x.SchemeName, x.CoverType,
                x.Insurer, x.OrderId, x.CreateDateTime
        from    (
                select  nullif(upper(trim(QuoteReference)), '')                                    as QuoteReference,
                        SessionId,
                        coalesce(`DateTime`, cast(CreateDateTime as string))    as PaymentDateTime,
                        case when `DateTime` is null then 1 else 0 end          as PaymentDateTimeIsLoadTime,
                        ClickCode,
                        QuoteStartDateTime,
                        PaymentType,
                        PaymentAmount,
                        PaymentStatus,
                        PaymentErrorCode,
                        PaymentErrorMessage,
                        SchemeName,
                        CoverType,
                        Insurer,
                        Order_ID                                                as OrderId,
                        CreateDateTime,
                        row_number() over (partition by upper(trim(QuoteReference)), SessionId,
                                                        coalesce(`DateTime`, cast(CreateDateTime as string)), ClickCode
                                           order by CreateDateTime desc) as rn
                from    ${ods.dlk}.vanpaymentjourney
                where   CreateDateTime >= date '${ods.cutoff_date}'
                ) x
        left join ${ods.target}.Quote q on q.SourceSystemId = 7 and q.SourceSystemQuoteReference = x.QuoteReference
        where   x.rn = 1
        ) x

## PolicyQuoteLink


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyQuoteLink                                                              */
/*  Description : The warehouse's quote-to-policy matches (Relay quote events, with the MFQ guid or HEQ */
/*              :  quotation they matched),                                                             */
/*              : as notebook 02 stg.QuoteToPolicyLink. One row per quote event (one duplicate in the s */
/*              : ource: latest kept).                                                                  */
/*              :                                                                                       */
/*              : Grain: one row per warehouse quote event                                              */
/*              : Business key: QuoteEventID                                                            */
/*              : History: event   Cut-off: QuoteEventDateTime                                          */
/*              : Source: edw.tbl_fact_policy_quotes                                                    */
/*              : Reads: edw.tbl_fact_policy_quotes, ods_v2.Policy, ods_v2.Quote                        */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyQuoteLink as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  x.QuoteEventID,
                p.PolicyId,
                qm.QuoteId                                                      as MfqQuoteId,
                x.QuoteEventDateTime,
                x.QuoteEventType,
                x.QuoteEventReportingTxnType,
                nullif(upper(trim(x.ClientCode)), '')                                              as ClientCode,
                nullif(upper(trim(x.PolicyCode)), '')                                              as PolicyCode,
                x.PolicyTypeCode,
                x.PolicyTypeGroup,
                x.PolicyStatusCode,
                x.RiskProfileBand,
                x.LeadType,
                x.OpportunityID,
                x.ChannelCall,
                x.ChannelWeb,
                x.ChannelXSell,
                x.ChannelOnLine,
                nullif(lower(trim(x.MFQQuoteQueryGUID)), '')                                       as MfqGuidKey,
                x.MFQCreatedDatetime,
                x.MFQQuoteType,
                x.MFQMatchType,
                x.MFQBrokerID,
                case when x.HEQQuotationID like '%@%' then ods_hash_id(x.HEQQuotationID, '')      -- some ids are email + timestamp
                     else nullif(upper(trim(x.HEQQuotationID)), '') end                            as HeqQuotationId,
                x.HEQCreatedDatetime,
                x.HEQQuoteType,
                x.HEQMatchType
        from    (
                select  *, row_number() over (partition by QuoteEventID order by CreateDate desc) as rn
                from    ${ods.edw}.tbl_fact_policy_quotes
                where   QuoteEventDateTime >= date '${ods.cutoff_date}'
                ) x
        left join ${ods.target}.Policy  p   on p.PolicyCode = upper(trim(x.PolicyCode))
        left join ${ods.target}.Quote   qm  on qm.SourceSystemId = 2 and qm.SourceSystemQuoteReference = nullif(lower(trim(x.MFQQuoteQueryGUID)), '')
        where   x.rn = 1
        ) x

## CallSession


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.CallSession                                                                  */
/*  Description : Genesys sessions (all media) with routing, outcome and durations. The customer's numb */
/*              : ers (ani / dnis) are salted                                                           */
/*              : PhoneKeys; agent names and from / to addresses are not taken. ParticipantName is kept */
/*              :  only for system / queue                                                              */
/*              : participants (it can be the customer's name on customer / external legs). The *IsMobi */
/*              : le flags say whether each                                                             */
/*              : number is an Irish mobile (national form 08...), set before hashing (notebook 04 cell */
/*              :  165 filters on it).                                                                  */
/*              :                                                                                       */
/*              : Grain: one row per Genesys conversation and session                                   */
/*              : Business key: ConversationId + SessionId                                              */
/*              : History: event   Cut-off: ConversationStartTime                                       */
/*              : Source: dlk.genesys_session_summary                                                   */
/*              : Reads: dlk.genesys_session_summary                                                    */
/******************************************************************************************************/
create or replace table ${ods.target}.CallSession as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  g.conversationId                                                as ConversationId,
                g.sessionId                                                     as SessionId,
                g.sessionIndex                                                  as SessionIndex,
                g.participantid                                                 as ParticipantId,
                g.conversationStartTime                                         as ConversationStartTime,
                g.conversationEndTime                                           as ConversationEndTime,
                g.sessionStartTime                                              as SessionStartTime,
                g.sessionEndTime                                                as SessionEndTime,
                g.mediaType                                                     as MediaType,
                g.messageType                                                   as MessageType,
                g.direction                                                     as Direction,
                g.originatingDirection                                          as OriginatingDirection,
                g.purpose                                                       as Purpose,
                g.queueId                                                       as QueueId,
                g.queueName                                                     as QueueName,
                case when lower(g.purpose) in ('customer', 'external') then null else g.participantName end
                                                                                as ParticipantName,
                g.campaignName                                                  as CampaignName,
                g.userId                                                        as AgentUserId,
                g.abandoned                                                     as Abandoned,
                g.agentAnswered                                                 as AgentAnswered,
                g.alertNoAnswer                                                 as AlertNoAnswer,
                g.offered                                                       as Offered,
                g.queueAnswered                                                 as QueueAnswered,
                g.transferred                                                   as Transferred,
                g.blindtransferred                                              as BlindTransferred,
                g.consulttransferred                                            as ConsultTransferred,
                ods_redact(g.transferToId)                                          as TransferToId,       -- external transfers carry a number
                g.transferToPurpose                                             as TransferToPurpose,
                g.disconnectType                                                as DisconnectType,
                g.wrapupCode                                                    as WrapupCode,
                g.wrapupCodeName                                                as WrapupCodeName,
                g.sessionDuration                                               as SessionDurationMs,
                g.totalAcdWaitDuration                                          as AcdWaitDurationMs,
                g.totalAgentAlertDuration                                       as AgentAlertDurationMs,
                g.totalAgentTalkDuration                                        as AgentTalkDurationMs,
                g.totalAgentHoldDuration                                        as AgentHoldDurationMs,
                g.totalAgentWrapupDuration                                      as AgentWrapupDurationMs,
                g.holdcount                                                     as HoldCount,
                g.callbackCount                                                 as CallbackCount,
                ods_hash_phone(replace(g.ani, 'tel:', ''))                          as AniPhoneKey,
                ods_hash_phone(replace(g.dnis, 'tel:', ''))                         as DnisPhoneKey,
                ods_hash_phone(replace(g.originatingDnis, 'tel:', ''))              as OriginatingDnisPhoneKey,
                case when regexp_replace(regexp_replace(regexp_replace(regexp_replace(cast(replace(g.ani, 'tel:', '') as string), '[^0-9]', ''), '^00353', '0'), '^353', '0'), '^(8[0-9]{8})$', '0$1') like '08%' then 1 else 0 end                           as AniIsMobile,
                case when regexp_replace(regexp_replace(regexp_replace(regexp_replace(cast(replace(g.dnis, 'tel:', '') as string), '[^0-9]', ''), '^00353', '0'), '^353', '0'), '^(8[0-9]{8})$', '0$1') like '08%' then 1 else 0 end                          as DnisIsMobile,
                case when regexp_replace(regexp_replace(regexp_replace(regexp_replace(cast(replace(g.originatingDnis, 'tel:', '') as string), '[^0-9]', ''), '^00353', '0'), '^353', '0'), '^(8[0-9]{8})$', '0$1') like '08%' then 1 else 0 end               as OriginatingDnisIsMobile
        from    ${ods.dlk}.genesys_session_summary g
        where   g.conversationStartTime >= date '${ods.cutoff_date}'
        ) x

## TelephonyCall


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.TelephonyCall                                                                */
/*  Description : The warehouse's call detail: call legs with work group, disposition and the MFQ / HEQ */
/*              :  quote and client the                                                                 */
/*              : warehouse matched each call to. Phone numbers are salted PhoneKeys; agent names are n */
/*              : ot taken. To be derived                                                               */
/*              : from dlk.genesys_session_summary with the DWH logic later.                            */
/*              :                                                                                       */
/*              : Grain: one row per call id and sequence                                               */
/*              : Business key: CallIDKey + SeqNo                                                       */
/*              : History: event   Cut-off: CallStartDT                                                 */
/*              : Source: edw.tbl_fact_telephony_call_detail (derived in the warehouse from Genesys; se */
/*              : e DWH Build)                                                                          */
/*              : Reads: edw.tbl_fact_telephony_call_detail                                             */
/******************************************************************************************************/
create or replace table ${ods.target}.TelephonyCall as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  t.CallIDKey,
                t.SeqNo,
                t.CallStartDT,
                t.CallConnectedDT,
                t.CallTerminatedDT,
                t.CallType,
                t.CallDirection,
                t.WorkGroup,
                t.Disposition,
                t.DispositionCode,
                t.WrapUpCodeName,
                t.MediaType,
                t.ConnectedDuration,
                t.HoldDuration,
                t.CallDuration,
                t.CampaignCall,
                t.CampaignName,
                t.CampaignWrapUpCode,
                t.AgentCode,
                t.WFPersonID                                                    as AgentPersonId,
                t.ExtTeamName,
                t.ExtDepartment,
                nullif(upper(trim(t.ClientCode)), '')                                              as ClientCode,
                t.OpportunityPolicyTypeGroup,
                t.NewBusIBFlag,
                t.NewBusOBFlag,
                t.RenewalsIBFlag,
                t.MTAIBFlag,
                nullif(lower(trim(t.MFQQuoteQueryGuid)), '')                                       as MfqGuidKey,
                t.MFQCreatedDatetime,
                t.MFQQuoteType,
                t.MFQMatchType,
                case when t.HEQQuotationID like '%@%' then ods_hash_id(t.HEQQuotationID, '')      -- some ids are email + timestamp
                     else nullif(upper(trim(t.HEQQuotationID)), '') end                            as HeqQuotationId,
                t.HEQCreatedDateTime,
                t.HEQMatchType,
                t.GenesysLeadID,
                ods_hash_phone(t.PhoneNumber)                                       as PhoneKey
        from    ${ods.edw}.tbl_fact_telephony_call_detail t
        where   t.CallStartDT >= date '${ods.cutoff_date}'
        ) x

## MessageEvent


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_xp_events                                                            */
/*  Description : Both Xtremepush feeds in one shape, contact details hashed.                           */
/*              :                                                                                       */
/*              : Reads: dlk.ext_xtremepushresults, dlk.ext_xtremepushresults_policy                    */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_xp_events as
select  'Quote' as SourceFeed, campaign_id, campaign_name, campaign_variant, profile_id, interaction_type, interaction_value,
        `timestamp`, PolicyCode, ClientCode, QuoteQueryGuid, workflow_node, MessageType, SourceFileDate, `$ExtractDateTime` as ExtractDateTime,
        ods_hash_email(email) as EmailKey, ods_hash_phone(mobile_number) as MobileKey, ods_hash_id(user_id, '') as UserIdKey
from    ${ods.dlk}.ext_xtremepushresults
where   `timestamp` >= date '${ods.cutoff_date}'
union all
select  'Policy', campaign_id, campaign_name, campaign_variant, profile_id, interaction_type, interaction_value,
        `timestamp`, PolicyCode, ClientCode, QuoteQueryGuid, workflow_node, MessageType, SourceFileDate, `$ExtractDateTime`,
        ods_hash_email(email), ods_hash_phone(mobile_number), ods_hash_id(user_id, '')
from    ${ods.dlk}.ext_xtremepushresults_policy
where   `timestamp` >= date '${ods.cutoff_date}'

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.MessageEvent                                                                 */
/*  Description : Xtremepush message events (sent / open / click / bounce ...) from the quote and polic */
/*              : y feeds, one row per event:                                                           */
/*              : the same event loaded from several files is kept once (latest file). Email, mobile an */
/*              : d the Xtremepush user id                                                              */
/*              : are salted hash keys.                                                                 */
/*              :                                                                                       */
/*              : Grain: one row per message event (feed, campaign, profile, interaction, time, value,  */
/*              : variant, node, message type)                                                          */
/*              : Business key: SourceFeed + CampaignId + ProfileId + InteractionType + EventDateTime + */
/*              :  InteractionValue + CampaignVariant + WorkflowNode + MessageType                      */
/*              : History: event   Cut-off: EventDateTime                                               */
/*              : Source: dlk.ext_xtremepushresults + dlk.ext_xtremepushresults_policy                  */
/*              : Reads: stg.odsv2_xp_events                                                            */
/******************************************************************************************************/
create or replace table ${ods.target}.MessageEvent as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  x.SourceFeed,
                x.campaign_id                                                   as CampaignId,
                x.campaign_name                                                 as CampaignName,
                x.campaign_variant                                              as CampaignVariant,
                x.profile_id                                                    as ProfileId,
                x.interaction_type                                              as InteractionType,
                ods_redact(x.interaction_value)                                     as InteractionValue,
                x.`timestamp`                                                   as EventDateTime,
                x.workflow_node                                                 as WorkflowNode,
                x.MessageType,
                nullif(upper(trim(x.PolicyCode)), '')                                              as PolicyCode,
                nullif(upper(trim(x.ClientCode)), '')                                              as ClientCode,
                nullif(lower(trim(x.QuoteQueryGuid)), '')                                          as QuoteGuidKey,
                x.EmailKey,
                x.MobileKey,
                x.UserIdKey,
                x.SourceFileDate
        from    (
                select  *,
                        row_number() over (partition by SourceFeed, campaign_id, profile_id, interaction_type, `timestamp`,
                                                        interaction_value, campaign_variant, workflow_node, MessageType
                                           order by SourceFileDate desc, ExtractDateTime desc) as rn
                from    ${ods.stg}.odsv2_xp_events
                ) x
        where   x.rn = 1
        ) x

## Campaign


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.Campaign                                                                     */
/*  Description : Campaign names versioned by date: each name a campaign id has carried, with the first */
/*              :  and last message event under                                                         */
/*              : it. NameCount > 1 means the campaign was renamed; NamesOverlap = 1 flags names used i */
/*              : n overlapping periods (then                                                           */
/*              : the dates do not separate them cleanly).                                              */
/*              :                                                                                       */
/*              : Grain: one row per campaign id and campaign name                                      */
/*              : Business key: CampaignId + CampaignName   Surrogate key: CampaignVersionId            */
/*              : History: versioned (ValidFrom / ValidTo = first / last event under that name)   Cut-o */
/*              : ff: none                                                                              */
/*              : Source: ods MessageEvent (Xtremepush campaigns)                                       */
/*              : Reads: ods_v2.MessageEvent                                                            */
/******************************************************************************************************/
create or replace table ${ods.target}.Campaign as
select  row_number() over (order by CampaignId, CampaignName)       as CampaignVersionId,
        x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  c.CampaignId,
                c.CampaignName,
                c.ValidFrom,
                c.ValidTo,
                c.EventCount,
                count(*) over (partition by c.CampaignId)                       as NameCount,
                case when lead(c.ValidFrom) over (partition by c.CampaignId order by c.ValidFrom, c.CampaignName) <= c.ValidTo
                       or lag(c.ValidTo) over (partition by c.CampaignId order by c.ValidFrom, c.CampaignName) >= c.ValidFrom
                     then 1 else 0 end                                          as NamesOverlap,
                case when row_number() over (partition by c.CampaignId order by c.ValidTo desc, c.CampaignName) = 1
                     then 1 else 0 end                                          as IsLatestName
        from    (
                select  CampaignId, CampaignName, min(EventDateTime) as ValidFrom, max(EventDateTime) as ValidTo, count(*) as EventCount
                from    ${ods.target}.MessageEvent
                where   CampaignId is not null
                group by CampaignId, CampaignName
                ) c
        ) x

## ConsentAction


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.ConsentAction                                                                */
/*  Description : Every marketing consent action recorded in Mercury (all history, not only the latest  */
/*              : per contact): channel,                                                                */
/*              : action, when, and the app that recorded it. The contact (email for channel 3, phone f */
/*              : or channels 1 / 2) is a                                                               */
/*              : salted EmailKey / PhoneKey, so ads can link consent to MFQ quotes and SCV keys by ema */
/*              : il point in time.                                                                     */
/*              :                                                                                       */
/*              : Grain: one row per Mercury consent action                                             */
/*              : Business key: ActionTrackId                                                           */
/*              : History: event (no cut-off; consent is a state, its full history is needed)   Cut-off */
/*              : : none                                                                                */
/*              : Source: dlk.merc_actiontracks + merc_servicecallstracks + merc_apps                   */
/*              : Reads: dlk.merc_actiontracks, dlk.merc_servicecallstracks, dlk.merc_apps              */
/******************************************************************************************************/
create or replace table ${ods.target}.ConsentAction as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  a.ActionTrackId,
                a.ActionId,
                a.ChannelId,
                case when a.ChannelId = 3 then 'Email' when a.ChannelId in (1, 2) then 'Phone' else 'Other' end
                                                                                as ContactType,
                a.`DateTime`                                                    as ConsentRecordedDateTime,
                a.EffectiveDateTime,
                a.ServiceCallsTrackId,
                a.BrokerId,
                s.AppId,
                p.Name                                                          as AppName,
                s.SubApp,
                s.isServiceExecuted                                             as IsServiceExecuted,
                case when a.ChannelId = 3 then ods_hash_email(a.Value) end          as EmailKey,
                case when a.ChannelId in (1, 2) then ods_hash_phone(a.Value) end    as PhoneKey
        from    ${ods.dlk}.merc_actiontracks            a
        left join ${ods.dlk}.merc_servicecallstracks    s   on s.ServiceCallsTrackId = a.ServiceCallsTrackId
        left join ${ods.dlk}.merc_apps                  p   on p.AppId = s.AppId
        ) x

## CdaMessage


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.CdaMessage                                                                   */
/*  Description : CDA (SMS / email) messages sent per client and policy: campaign, batch, status and pr */
/*              : ovider response, and the                                                              */
/*              : message type. The body is read only to classify it (7 patterns covered all 116,711 me */
/*              : ssages of Jun-Aug 2026); it is                                                        */
/*              : not held (it carries names). Every message carries its policy number or client code i */
/*              : n its own column, so no code                                                          */
/*              : is read from the text. 99 = the SMS engine's own daily test message. Contact and emai */
/*              : l are salted keys.                                                                    */
/*              :                                                                                       */
/*              : Grain: one row per CDA message                                                        */
/*              : Business key: MessageId                                                               */
/*              : History: event   Cut-off: SentDateTime                                                */
/*              : Source: dlk.sql07cda_messages (read from dlk_bak_20260914 until it lands in dlk)      */
/*              : Reads: dlk_bak_20260914.sql07cda_messages                                             */
/******************************************************************************************************/
create or replace table ${ods.target}.CdaMessage as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  m.MessageId,
                case
                     when m.RenderedMessage like '%FQ Chase%' then 1
                     when m.RenderedMessage like '%emailed important correspondence%' then 2
                     when m.RenderedMessage like '%thanks for choosing us%' then 3
                     when m.RenderedMessage like '%thank you for choosing Chill Insurance. We have issued%' then 4
                     when m.RenderedMessage like '%renewal is due tomorrow%' then 5
                     when m.RenderedMessage like '%renewal is now due%' then 6
                     when m.RenderedMessage like '%SMS engine is OK%' then 99
                     else 0 end                                                 as MessageTypeId,
                case
                     when m.RenderedMessage like '%FQ Chase%' then 'MFQ chase'
                     when m.RenderedMessage like '%emailed important correspondence%' then 'Correspondence emailed'
                     when m.RenderedMessage like '%thanks for choosing us%' then 'Welcome'
                     when m.RenderedMessage like '%thank you for choosing Chill Insurance. We have issued%' then 'Policy issued'
                     when m.RenderedMessage like '%renewal is due tomorrow%' then 'Renewal due tomorrow'
                     when m.RenderedMessage like '%renewal is now due%' then 'Renewal now due'
                     when m.RenderedMessage like '%SMS engine is OK%' then 'SMS engine test'
                     else 'Unclassified' end                                    as MessageType,
                m.DailyCampaignId,
                m.BatchId,
                m.SentDateTime,
                m.MessageStatus,
                m.ProviderResponse,
                m.ProviderResponseDateTime,
                m.UTCDateAdded,
                m.UTCDateUpdated,
                m.Priority,
                nullif(upper(trim(m.ClientCode)), '')                                              as ClientCode,
                nullif(upper(trim(m.PolicyNumber)), '')                                            as PolicyCode,
                m.IsSendViaChillPush,
                case when m.Contact like '%@%' then ods_hash_email(m.Contact) else ods_hash_phone(m.Contact) end
                                                                                as ContactKey,
                ods_hash_email(m.Email)                                             as EmailKey
        from    ${ods.dlk_cda}.sql07cda_messages m
        where   m.SentDateTime >= date '${ods.cutoff_date}'
        ) x

## WebPageView


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_ga4_quote_sessions                                                   */
/*  Description : GA4 sessions with at least one motor or home quote reference.                         */
/*              :                                                                                       */
/*              : Reads: dlk.ga4_page_views_daily                                                       */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_ga4_quote_sessions as
select  distinct SessionID
from    ${ods.dlk}.ga4_page_views_daily
where   EventDate >= date '${ods.cutoff_date}'
and     (nullif(trim(MotorQuoteReference), '') is not null or nullif(trim(HomeQuoteReference), '') is not null)

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.WebPageView                                                                  */
/*  Description : GA4 page views in sessions that carry a motor or home quote reference (the sessions t */
/*              : he assets use: device,                                                                */
/*              : source, pages around a quote). PagePathCleaned only; the raw path with query string i */
/*              : s not taken.                                                                          */
/*              :                                                                                       */
/*              : Grain: one row per GA4 page view (client, click order, effective date, event time)    */
/*              : Business key: ClientID + ClickOrder + EffectiveDate + EventDateTime                   */
/*              : History: event   Cut-off: EventDate                                                   */
/*              : Source: dlk.ga4_page_views_daily                                                      */
/*              : Reads: dlk.ga4_page_views_daily, stg.odsv2_ga4_quote_sessions                         */
/******************************************************************************************************/
create or replace table ${ods.target}.WebPageView as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  g.ClientID,
                g.ClickOrder,
                g.EffectiveDate,
                g.EventDate,
                g.EventDateTime,
                g.EventName,
                g.SessionID,
                g.SessionDuration,
                g.SessionEndDateTime,
                g.SessionGroup,
                g.SessionCount,
                g.GASource,
                g.GAMedium,
                g.DeviceCategory,
                g.PageTitle,
                g.PagePathCleaned,
                g.UserPseudoID,
                g.CampaignID,
                g.CampaignName,
                g.Keyword,
                g.PageViewDuration,
                g.MotorQuoteID,
                nullif(upper(trim(g.MotorQuoteReference)), '')                                     as MotorQuoteReference,
                nullif(upper(trim(g.HomeQuoteReference)), '')                                      as HomeQuoteReference,
                g.OrgID
        from    ${ods.dlk}.ga4_page_views_daily     g
        join    ${ods.stg}.odsv2_ga4_quote_sessions s   on s.SessionID = g.SessionID
        where   g.EventDate >= date '${ods.cutoff_date}'
        ) x

## WebQuoteClick


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.WebQuoteClick                                                                */
/*  Description : GA4 clicks that carry an MFQ quote guid (the five MFQ-QuoteQueryGUID columns turned i */
/*              : nto rows), with source,                                                               */
/*              : medium, campaign and device, as ODS_Build_QuoteJourney GAQuoteClicks.                 */
/*              :                                                                                       */
/*              : Grain: one row per MFQ guid and GA click                                              */
/*              : Business key: GuidKey + ClientID + ClickOrder + EffectiveDate                         */
/*              : History: event   Cut-off: WorkDate                                                    */
/*              : Source: edw.tbl_fact_ga_clicks_ga4                                                    */
/*              : Reads: edw.tbl_fact_ga_clicks_ga4, ods_v2.Quote                                       */
/******************************************************************************************************/
create or replace table ${ods.target}.WebQuoteClick as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  distinct
                q.QuoteId,
                nullif(lower(trim(g.guid)), '')                                                    as GuidKey,
                g.ClientID,
                g.ClickOrder,
                g.EffectiveDate,
                g.WorkDate,
                g.WorkDateTime                                                  as WorkDateTimeUtc,
                g.GASource,
                g.GAMedium,
                g.Campaign,
                g.DeviceCategory,
                g.FirstDevice,
                ods_redact(g.PagePath)                                              as PagePath
        from    (
                select  c.*, s.guid
                from    ${ods.edw}.tbl_fact_ga_clicks_ga4 c
                lateral view stack(5, `MFQ-QuoteQueryGUID1`, `MFQ-QuoteQueryGUID2`, `MFQ-QuoteQueryGUID3`,
                                      `MFQ-QuoteQueryGUID4`, `MFQ-QuoteQueryGUID5`) s as guid
                where   c.WorkDate >= date '${ods.cutoff_date}'
                ) g
        left join ${ods.target}.Quote q on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = nullif(lower(trim(g.guid)), '')
        where   nullif(trim(g.guid), '') is not null
        ) x

## WebRetrieveHit


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.WebRetrieveHit                                                               */
/*  Description : Car retrieve-page hits with ?reference= and the reference they carried (paths as note */
/*              : book 05 cells 11-12). Raw                                                             */
/*              : hits only: resolving each hit to the quote it opened (first quote with that reference */
/*              :  created 5 s before to 30 min                                                         */
/*              : after the hit; references are reused after 30-60 days) is a derived relationship and  */
/*              : belongs in ads.                                                                       */
/*              :                                                                                       */
/*              : Grain: one row per retrieve-page hit with a ?reference= (client, click order, time)   */
/*              : Business key: ClientID + ClickOrder + HitDateTimeUtc                                  */
/*              : History: event   Cut-off: HitDateTimeUtc                                              */
/*              : Source: edw.tbl_fact_ga_clicks_ga4                                                    */
/*              : Reads: edw.tbl_fact_ga_clicks_ga4                                                     */
/******************************************************************************************************/
create or replace table ${ods.target}.WebRetrieveHit as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  ClientID,
                ClickOrder,
                HitDateTimeUtc,
                ods_redact(PagePath)                                                as PagePath,
                Reference
        from    (
                select  ClientID,
                        ClickOrder,
                        WorkDateTime                                            as HitDateTimeUtc,
                        PagePath,
                        nullif(upper(trim(regexp_extract(regexp_extract(PagePath, '\\?([^#]*)', 1),
                                                         '(?i)(?:^|&)reference=([^&]*)', 1))), '')
                                                                                as Reference,
                        row_number() over (partition by ClientID, ClickOrder, WorkDateTime order by EffectiveDate desc) as rn
                from    ${ods.edw}.tbl_fact_ga_clicks_ga4
                where   lower(PagePath) like '%reference=%'
                and     WorkDate >= date '${ods.cutoff_date}'
                and     lower(concat(case when PagePath like '/%' then '' else '/' end, regexp_extract(PagePath, '^([^?#]*)', 1))) in
                        ('/retrieve-quote/', '/retrieve', '/retrieve/', '/customer-care/motor/retrieve-my-quote/', '/car-insurance/')
                ) h
        where   rn = 1
        and     Reference is not null
        and     Reference not in ('GDPR', 'NONE', 'NAN', 'NULL')
        ) x

## SourceCustomer


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_source_customer_address                                              */
/*  Description : One address (lines 1 and 2 as a salted key, Eircode area) per source customer, the hi */
/*              : ghest address id.                                                                     */
/*              :                                                                                       */
/*              : Reads: pii.addresses_assembled                                                        */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_source_customer_address as
select  SourceSystemID, SubSourceSystemID, cast(SourceCustomerID as string) as SourceCustomerID,
        ods_hash_id(concat_ws(' ', AddressLine1, AddressLine2), '')         as AddressKey,
        ods_eircode_area(Postcode)                                      as EircodeArea
from    (
        select  *, row_number() over (partition by SourceSystemID, SubSourceSystemID, SourceCustomerID
                                      order by RapierAddressId desc) as rn
        from    ${ods.pii}.addresses_assembled
        ) a
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.SourceCustomer                                                               */
/*  Description : Every customer as each source system holds it (Relay client, MFQ / QuotationStorage d */
/*              : river, travel customer ...),                                                          */
/*              : from the SCV pii tables, reduced to salted keys for matching and corroboration in ads */
/*              : : email, phone, address lines                                                         */
/*              : 1+2 (as the warehouse's address match), name + date of birth, surname + date of birth */
/*              : . Nothing personal is held:                                                           */
/*              : no names, dates of birth, addresses or contact details. RapierCustomerId (= ChillSour */
/*              : ceCustomerKey) and the SCV                                                            */
/*              : address keys join to the SCV tables. Assumes one pii row per source customer, and add */
/*              : resses joined on the same                                                             */
/*              : three columns (the pii schemas are closed to profiling: confirm on the first run).    */
/*              :                                                                                       */
/*              : Grain: one row per source customer (source system, sub source, source customer id)    */
/*              : Business key: SourceSystemId + SubSourceSystemId + SourceSystemCustomerId             */
/*              : History: current   Cut-off: none                                                      */
/*              : Source: pii.customer_data_assembled + pii.addresses_assembled (the SCV source custome */
/*              : rs)                                                                                   */
/*              : Reads: pii.customer_data_assembled, stg.odsv2_source_customer_address                 */
/******************************************************************************************************/
create or replace table ${ods.target}.SourceCustomer as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  c.SourceSystemId,
                c.SubSourceSystemId,
                cast(c.SourceSystemCustomerId as string)                        as SourceSystemCustomerId,
                c.RapierCustomerId,
                c.SCV_Address_Key,
                c.ChillSourceAddressKey,
                c.CustomerGender,
                ods_hash_email(c.Email)                                             as EmailKey,
                ods_hash_phone(c.CustomerPhone)                                     as PhoneKey,
                a.AddressKey,
                coalesce(a.EircodeArea, ods_eircode_area(c.Postcode))               as EircodeArea,
                case when c.CustomerDateOfBirth is not null and nullif(trim(c.CustomerSurName), '') is not null
                     then ods_hash_id(concat_ws('|', c.CustomerFirstName, c.CustomerSurName, cast(cast(c.CustomerDateOfBirth as date) as string)), '')
                end                                                             as NameDobKey,
                case when c.CustomerDateOfBirth is not null and nullif(trim(c.CustomerSurName), '') is not null
                     then ods_hash_id(concat_ws('|', c.CustomerSurName, cast(cast(c.CustomerDateOfBirth as date) as string)), '')
                end                                                             as SurnameDobKey
        from    (
                select  *, row_number() over (partition by SourceSystemId, SubSourceSystemId, SourceSystemCustomerId
                                              order by RapierCustomerId desc) as rn
                from    ${ods.pii}.customer_data_assembled
                ) c
        left join ${ods.stg}.odsv2_source_customer_address a
                on  a.SourceSystemID = c.SourceSystemId and a.SubSourceSystemID = c.SubSourceSystemId
                and a.SourceCustomerID = cast(c.SourceSystemCustomerId as string)
        where   c.rn = 1
        ) x

## Calendar


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.Calendar                                                                     */
/*  Description : Chill's calendar: weekday, working day, holiday, week (Sunday-Saturday and Monday-Sun */
/*              : day), financial year.                                                                 */
/*              :                                                                                       */
/*              : Grain: one row per day                                                                */
/*              : Business key: DayDate                                                                 */
/*              : History: current   Cut-off: none                                                      */
/*              : Source: edw.tbl_ref_calendarlookup                                                    */
/*              : Reads: edw.tbl_ref_calendarlookup                                                     */
/******************************************************************************************************/
create or replace table ${ods.target}.Calendar as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  DayDate,
                DayDatetime,
                DayOfTheWeek,
                DayOfTheWeekDesc,
                DayOfTheMonth,
                MonthOfTheYear,
                DayMonthCode,
                DayOfTheYear,
                DayYear,
                WorkingDayOfTheMonth,
                WorkingDayOfTheYear,
                WorkingDayOfTheYearMonSat,
                WorkingDay,
                WorkingDayMonSat,
                BetweenMonFry,
                BetweenMonSat,
                Holiday,
                IsDST,
                WeekOfTheYear,
                WeekOfTheYearMonSun,
                FinancialYear
        from    ${ods.edw}.tbl_ref_calendarlookup
        ) x

## Checks


In [ ]:
%%pyspark
# Checks for the tables this notebook built. All are checked and logged to OdsBuildLog; the cell then fails if
# any check failed, naming the table and the likely cause.
import json, re
from functools import reduce
from pyspark.sql import functions as F
SPEC = json.loads(r"""[{"table": "Policy", "bk": ["PolicyCode"], "sk": "PolicyId", "nullable": [], "allow": []}, {"table": "PolicyVersion", "bk": ["PolicyCode", "RecordVersion"], "sk": "PolicyVersionId", "nullable": [], "allow": []}, {"table": "PolicyTravel", "bk": ["MapfrePolicyId"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyTravelTransaction", "bk": ["TravelTransactionKey"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyMotorProposer", "bk": ["PolicyCode", "RecordVersion"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyMotorVehicle", "bk": ["PolicyCode", "RecordVersion"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyHome", "bk": ["PolicyCode", "RecordVersion"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicySale", "bk": ["PolicyCode", "ReportingSaleDate"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyRenewal", "bk": ["PolicyCode", "LYPolicyRenewDateAdj"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyMonthEnd", "bk": ["PolicyCode", "EffectiveDate"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyCancellation", "bk": ["PolicyCancellationId"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyTransaction", "bk": ["PolicyCode", "PostingDate", "TransSeq", "ArCode", "ArNumber", "ArReference", "ClientTransType", "ClientAmount"], "sk": "PolicyTransactionId", "nullable": [], "allow": []}, {"table": "PolicyMta", "bk": ["PolicyCode", "HistorySeq", "HistoryType"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyClaimMotor", "bk": ["PolicyCode", "ClaimDateRaw", "ClaimType", "ClaimValue", "ClaimSlot"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyClaimHome", "bk": ["HomeClaimId"], "sk": null, "nullable": [], "allow": []}, {"table": "ClientPortfolio", "bk": ["PortfolioId", "PolicyCode"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyDocumentEvent", "bk": ["EventId", "DocumentId", "PolicyCode"], "sk": null, "nullable": [], "allow": []}, {"table": "RenewalPortalEvent", "bk": ["EventType", "EventId"], "sk": null, "nullable": [], "allow": []}, {"table": "DocumentWorkflowStatus", "bk": ["Pk", "Rk"], "sk": null, "nullable": [], "allow": []}, {"table": "DocumentUpload", "bk": ["Pk", "Rk"], "sk": null, "nullable": [], "allow": []}, {"table": "DocumentChaseHome", "bk": ["FileDate", "PolicyCode"], "sk": null, "nullable": [], "allow": []}, {"table": "DocumentChaseMotorVan", "bk": ["FileDate", "PolicyCode"], "sk": null, "nullable": [], "allow": []}, {"table": "Quote", "bk": ["SourceSystemId", "SourceSystemQuoteReference"], "sk": "QuoteId", "nullable": [], "allow": []}, {"table": "QuoteMotorPrice", "bk": ["GuidKey", "Insurer"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteMotorPayment", "bk": ["PaymentId"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteMotorStep", "bk": ["ClientStepId"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteMotorBotSession", "bk": ["MonSessionId"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteMotor", "bk": ["GuidKey"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteMotorSummary", "bk": ["GuidKey"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteMotorDriver", "bk": ["SourceSystemId", "SourceSystemQuoteReference", "DriverPRN"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteMotorVehicle", "bk": ["SourceSystemId", "SourceSystemQuoteReference"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteHome", "bk": ["SourceSystemId", "SourceSystemQuoteReference"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyHomeQsEvent", "bk": ["HomeRiskId"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteHomePrice", "bk": ["ResponseQuoteId"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteHomePaymentRequest", "bk": ["GlobalPaymentReference"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteHomePaymentResponse", "bk": ["PaymentResponseId"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteHomeSale", "bk": ["SaleRecordId"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteTravel", "bk": ["SourceSystemQuoteReference"], "sk": null, "nullable": [], "allow": ["MarketingConsent_Email", "MarketingConsent_SMS"]}, {"table": "QuoteVan", "bk": ["RowKey"], "sk": null, "nullable": [], "allow": ["IsTestEmail"]}, {"table": "QuoteVanJourney", "bk": ["QuoteReference", "SessionId", "EventTimestamp", "ClickCode"], "sk": null, "nullable": ["SessionId"], "allow": ["InvalidEmailAddress", "InvalidDob"]}, {"table": "QuoteVanPayment", "bk": ["QuoteReference", "SessionId", "PaymentDateTime", "ClickCode"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyQuoteLink", "bk": ["QuoteEventID"], "sk": null, "nullable": [], "allow": []}, {"table": "CallSession", "bk": ["ConversationId", "SessionId"], "sk": null, "nullable": [], "allow": ["AniIsMobile", "DnisIsMobile", "OriginatingDnisIsMobile"]}, {"table": "TelephonyCall", "bk": ["CallIDKey", "SeqNo"], "sk": null, "nullable": [], "allow": []}, {"table": "MessageEvent", "bk": ["SourceFeed", "CampaignId", "ProfileId", "InteractionType", "EventDateTime", "InteractionValue", "CampaignVariant", "WorkflowNode", "MessageType"], "sk": null, "nullable": ["ProfileId", "InteractionValue", "CampaignVariant", "WorkflowNode", "MessageType", "CampaignId"], "allow": []}, {"table": "Campaign", "bk": ["CampaignId", "CampaignName"], "sk": "CampaignVersionId", "nullable": [], "allow": []}, {"table": "ConsentAction", "bk": ["ActionTrackId"], "sk": null, "nullable": [], "allow": []}, {"table": "CdaMessage", "bk": ["MessageId"], "sk": null, "nullable": [], "allow": []}, {"table": "WebPageView", "bk": ["ClientID", "ClickOrder", "EffectiveDate", "EventDateTime"], "sk": null, "nullable": [], "allow": []}, {"table": "WebQuoteClick", "bk": ["GuidKey", "ClientID", "ClickOrder", "EffectiveDate"], "sk": null, "nullable": [], "allow": []}, {"table": "WebRetrieveHit", "bk": ["ClientID", "ClickOrder", "HitDateTimeUtc"], "sk": null, "nullable": [], "allow": []}, {"table": "SourceCustomer", "bk": ["SourceSystemId", "SubSourceSystemId", "SourceSystemCustomerId"], "sk": null, "nullable": [], "allow": []}, {"table": "Calendar", "bk": ["DayDate"], "sk": null, "nullable": [], "allow": []}]""")
PII_NAME = re.compile(r"(e_?mail|forename|first_?name|surname|last_?name|full_?name|second_?name|phone|mobile|telephone|post_?code|eircode(?!_?area)|address_?line|risk_?address|postal_?address|dob$|date_?of_?birth|licen[cs]e_?num|registration_?no|reg_?number|registration_number|^ani$|^dnis$|originatingdnis|^payer_|iban|^bic|^contact$|rendered_?message)", re.I)
EMAIL_RE = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
MOBILE_RE = r"(^|[^0-9A-Za-z-])(\+?353 ?|0)8[35-9][ -]?[0-9]{3}[ -]?[0-9]{4}([^0-9A-Za-z-]|$)"   # stands alone, not inside an id
rows, failures = [], []
for s in SPEC:
    t = f"{target_schema}.{s['table']}"
    df = spark.table(t)
    bk = s["bk"]
    must = [c for c in bk if c not in s["nullable"]]
    agg = df.agg(F.count(F.lit(1)).alias("n"),
                 F.countDistinct(F.struct(*bk)).alias("d_bk"),
                 F.sum(F.when(reduce(lambda a, b: a | b, [F.col(c).isNull() for c in must] or [F.lit(False)]), 1).otherwise(0)).alias("null_bk"),
                 *( [F.countDistinct(s["sk"]).alias("d_sk")] if s["sk"] else [] )).first()
    pii_cols = [c for c in df.columns if PII_NAME.search(c) and not c.endswith("Key") and c not in s["allow"]]
    text_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]   # hashes never match
    hits = {}
    if text_cols:
        found = df.select(*text_cols).limit(1000000).agg(
            *[F.sum((F.col(c).rlike(EMAIL_RE) | F.col(c).rlike(MOBILE_RE)).cast("int")).alias(c) for c in text_cols]).first().asDict()
        hits = {c: v for c, v in found.items() if v}
    problems, advice = [], []
    keys = " + ".join(bk)
    if agg["n"] == 0:
        problems.append("empty")
        advice.append("the table is empty: its source is empty, or a filter (cut-off date, join) removed every row")
    if agg["d_bk"] != agg["n"]:
        problems.append(f"{agg['n'] - agg['d_bk']:,} duplicate business keys")
        advice.append(f"several rows share a key ({keys}): the source sent the same record more than once, or the "
                      f"dedupe rule in the cell no longer holds. Examples: select {', '.join(bk)}, count(*) from {t} "
                      f"group by {', '.join(bk)} having count(*) > 1")
    if agg["null_bk"]:
        problems.append(f"{agg['null_bk']:,} null business keys")
        advice.append(f"key columns ({', '.join(must)}) are empty on some rows: if that is legitimate for a column, add it "
                      f"to key_nullable for this table in the config; otherwise the source changed")
    if s["sk"] and agg["d_sk"] != agg["n"]:
        problems.append("surrogate key not unique")
        advice.append("the surrogate key repeats: the business key is not unique (see above)")
    if pii_cols:
        problems.append(f"PII-named columns {pii_cols}")
        advice.append("columns named like personal data: hash them (HASH_*) or leave them out in the config")
    if hits:
        problems.append(f"email/mobile values in {hits}")
        advice.append("emails / mobile numbers inside these columns: wrap the column in REDACT() (or hash it) in the config")
    rows.append((batch_id, s["table"], agg["n"], agg["n"] - agg["d_bk"], agg["null_bk"], "; ".join(problems) or "ok"))
    print(f"{s['table']:<28} {agg['n']:>13,}  {'; '.join(problems) or 'ok'}")
    if problems:
        failures.append(f"{s['table']}: " + " | ".join(advice))
log = spark.createDataFrame(rows, "OdsBatchId string, TableName string, RowCount long, DuplicateKeys long, NullKeys long, Result string")
log.withColumn("CheckedAt", F.current_timestamp()).write.mode("append").saveAsTable(f"{target_schema}.OdsBuildLog")
if failures:
    ods_run_log("Full", "failed", "; ".join(f.split(":")[0] for f in failures))
    raise AssertionError("ods checks failed:\n- " + "\n- ".join(failures))
ods_run_log("Full", "succeeded")
